# Low-utilization containers and idle times

Low-utilization containers, idle ratios, pre- and post-execution idle times and the IR2 timeouts (Tables 7 and 8, Figures 9-11, Sections 4.1-4.2). Run by `run_manuscript.py`; outputs in `05_Results/section4/execution_runs/03_idle`.

## Setup

In [ ]:
# Locate the repository root and load the folder constants.
from pathlib import Path
import sys
_start = Path.cwd().resolve()
_root = next((p for p in [_start,*_start.parents] if (p/'project_paths.py').is_file()), None)
if _root is None:
    raise RuntimeError('Run from inside the repository.')
for _p in [_root,_root/'04_Analysis/revision',_root/'02_Preprocessing']:
    sys.path.insert(0,str(_p))
from project_paths import *
install_data_io_guard()
PROJECT_ROOT = REVISION_CODE
ensure_dirs()
# Execution scope and output capture (no change to policy calculations).
import json as _mf_json
import re as _mf_re
import numpy as _mf_np
import pandas as _mf_pd
import matplotlib.pyplot as _mf_plt
from matplotlib.figure import Figure as _mf_Figure
_MF_OUTPUT = ROOT / '05_Results/section4/execution_runs/03_idle'
_MF_OUTPUT.mkdir(parents=True,exist_ok=True)
_MF_CAPTURE = _MF_OUTPUT / 'capture'
_MF_CAPTURE.mkdir(exist_ok=True)
_MF_MODE = 'corrected'
_MF_CELL = 0
_MF_ERRORS = []
PROCESSED_DATA = ROOT / '03_ProcessedData'
RESULTS_SECTION3 = _MF_OUTPUT
RESULTS_SECTION4 = _MF_OUTPUT
RESULTS_REVISION = _MF_OUTPUT
_MF_REAL_READ_CSV = _mf_pd.read_csv
def _mf_read_csv(*args,**kwargs):
    try:
        value = _MF_REAL_READ_CSV(*args,**kwargs)
    except UnicodeDecodeError:
        # Encoding fallback for cp949 CSV files (I/O only).
        value = _MF_REAL_READ_CSV(*args,**{**kwargs,"encoding":"cp949"})
    return value
_mf_pd.read_csv = _mf_read_csv
_mf_orig_savefig = _mf_Figure.savefig
def _mf_savefig(self,fname,*args,**kwargs):
    if isinstance(fname,(str,Path)):
        target=Path(fname)
        if not target.is_absolute() or not target.resolve().is_relative_to(_MF_OUTPUT):
            target = _MF_OUTPUT / target.name
        target.parent.mkdir(parents=True,exist_ok=True)
        fname=target
    return _mf_orig_savefig(self,fname,*args,**kwargs)
_mf_Figure.savefig = _mf_savefig
def _mf_capture_cell(cell_number):
    import warnings as _mf_warnings
    with _mf_warnings.catch_warnings():
        _mf_warnings.simplefilter('ignore')
        for name,value in list(globals().items()):
            if name.startswith('_'):continue
            if isinstance(value,_mf_pd.DataFrame) and len(value) <= 1000 and len(value.columns) <= 80:
                value.to_csv(_MF_CAPTURE/f'cell{cell_number:02d}_{name}.csv',index=True)
            elif isinstance(value,_mf_pd.Series) and len(value) <= 1000:
                value.to_frame().to_csv(_MF_CAPTURE/f'cell{cell_number:02d}_{name}.csv',index=True)
        for no in _mf_plt.get_fignums():
            figure=_mf_plt.figure(no)
            if not figure.axes:continue
            path=_MF_CAPTURE/f'cell{cell_number:02d}_figure{no}.png'
            _mf_orig_savefig(figure,path,dpi=110,bbox_inches='tight')
            lines=[]
            for axno,axis in enumerate(figure.axes):
                for lineno,line in enumerate(axis.get_lines()):
                    x,y=line.get_data()
                    if len(x)!=len(y) or len(x)>100000:continue
                    if len(x):
                        lines.extend((axno,lineno,line.get_label(),i,a,b) for i,(a,b) in enumerate(zip(x,y)))
            if lines:
                _mf_pd.DataFrame(lines,columns=['axis','line','label','point','x','y']).to_csv(path.with_suffix('.csv'),index=False)


# Export the actual plotted figures before notebook display closes them.
_MF_ORIG_SHOW=_mf_plt.show
_MF_FIG_COUNTER={}
def _mf_show(*args,**kwargs):
    for _number in _mf_plt.get_fignums():
        _figure=_mf_plt.figure(_number)
        if not _figure.axes:continue
        _MF_FIG_COUNTER[_MF_CELL]=_MF_FIG_COUNTER.get(_MF_CELL,0)+1
        _stem=f'cell{_MF_CELL:02d}_plot{_MF_FIG_COUNTER[_MF_CELL]:02d}'
        _folder=_MF_OUTPUT/'figures';_folder.mkdir(exist_ok=True)
        _mf_orig_savefig(_figure,_folder/(_stem+'.png'),dpi=150,bbox_inches='tight')
        _mf_orig_savefig(_figure,_folder/(_stem+'.pdf'),bbox_inches='tight')
    return _MF_ORIG_SHOW(*args,**kwargs)
_mf_plt.show=_mf_show


In [ ]:
_MF_CELL=2
# # 구글드라이브 연결
# from google.colab import drive



In [ ]:
_mf_capture_cell(2)


In [ ]:
_MF_CELL=3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns
default_colors = plt.rcParams['axes.prop_cycle'].by_key()['color'] # Matplotlib 기본 색상 가져오기
from pathlib import Path
import pyarrow as pa
import pyarrow.parquet as pq
from datetime import datetime
import time
import re
from functools import reduce

import warnings
warnings.filterwarnings('ignore')
import matplotlib.cbook
from matplotlib.ticker import LogLocator, LogFormatterMathtext, NullFormatter

import os

plt.rcParams['font.family'] = 'sans-serif'  # 기본 sans-serif 폰트 사용
plt.rcParams['font.sans-serif'] = ['Arial', 'Liberation Sans', 'DejaVu Sans']  # 대체 폰트
plt.rcParams['axes.unicode_minus'] = False  # 음수 기호 문제 해결


In [ ]:
_mf_capture_cell(3)


In [ ]:
_MF_CELL=4
# ============================================
# Low Util 기준 변수
# ============================================

CPU_IDLE_0GPU = 10
MEM_IDLE_0GPU = 15

CPU_IDLE_GPU  = 10
MEM_IDLE_GPU  = 20
GPU_IDLE_GPU  = 0  # GPU==0을 idle로 본다 (당신 기준)


In [ ]:
_mf_capture_cell(4)


In [ ]:
_MF_CELL=5
# 폰트 사이즈 조절
plt.rcParams['font.size'] = 15  # 모든 글자의 크기를 동일하게 설정
plt.rcParams['axes.titlesize'] = 15  # 그래프 제목
plt.rcParams['axes.labelsize'] = 15  # 축 레이블
plt.rcParams['xtick.labelsize'] = 15  # x축 눈금
plt.rcParams['ytick.labelsize'] = 15  # y축 눈금
plt.rcParams['lines.linewidth'] = 3   # 선 그래프 굵기 조절


In [ ]:
_mf_capture_cell(5)


In [ ]:
_MF_CELL=6
# ==============================================================================
# 0) 공통: 환경 판별 + BASE 설정 (코랩/로컬)
# ==============================================================================

from pathlib import Path
import os

def is_colab():
    return Path("/content").exists() and (
        "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
    )

COLAB_BASE = Path(str(PROCESSED_DATA))
LOCAL_BASE = Path(str(PROCESSED_DATA))

BASE = COLAB_BASE if is_colab() else LOCAL_BASE

# (선택) 코랩이면 드라이브 마운트
if is_colab():
    from google.colab import drive
    if not Path("/content/gdrive").exists():
        drive.mount("/content/gdrive")

print("✅ Running on:", "Colab" if is_colab() else "Local")
print("✅ BASE =", BASE)

# ==============================================================================
# 1) 라이브러리 (필요한 것만 우선)
# ==============================================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# 코랩에서만 매직 사용 (로컬 VSCode에선 에러 방지)
if is_colab():
    get_ipython().run_line_magic("matplotlib", "inline")

import warnings
warnings.filterwarnings("ignore")

# 폰트/마이너스 설정
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Arial", "Liberation Sans", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

# ==============================================================================
# 2) 데이터 로드 (UsageLog)
# ==============================================================================

df_UsageLog = pd.read_csv(BASE / "UsageLog20230401_20251130.csv")

# df_3는 이미 0.x GPU를 제거한 상태
df_3 = pd.read_parquet(BASE / "TimeSeries20240926_20251130.parquet")
print("✅ df_selected loaded:", df_3.shape)
print("✅ columns:", list(df_3.columns))



# 전처리된 파일을 읽어들여서 실행한다.



In [ ]:
_mf_capture_cell(6)


In [ ]:
_MF_CELL=7
df_3.groupby('gpu_allocated(group)')['gpu_allocated(group)'].value_counts()


In [ ]:
_mf_capture_cell(7)


In [ ]:
_MF_CELL=8
# 컬럼 값 변경
df_3 = df_3.rename(columns={
    "cpu_util(%)": "CPU",
    "mem_util(%)": "Memory",
    "gpu_util(%)": "GPU"
})

# gpu_allocated(group) 값 변경
df_3['gpu_allocated(group)'] = df_3['gpu_allocated(group)'].replace({
    '0.0_GPU': '0 GPU',
    '1.0_GPU': '1 GPU',
    '2.0_GPU': '2 GPU',
    '3.0_GPU': '3 GPU',
    '4.0_GPU': '4 GPU'
})


In [ ]:
_mf_capture_cell(8)


In [ ]:
_MF_CELL=9
df_3_gpu = df_3[(df_3['gpu_allocated(group)'] != '0 GPU') & (~df_3['gpu_allocated(group)'].isnull())]
df_3_cpu = df_3[(df_3['gpu_allocated(group)'] == '0 GPU') & (~df_3['gpu_allocated(group)'].isnull())]

# df_3_gpu.to_csv(str(PROCESSED_DATA),sep=",",na_rep='NaN', index=False)
# df_3_cpu.to_csv(str(PROCESSED_DATA),sep=",",na_rep='NaN', index=False)


In [ ]:
_mf_capture_cell(9)


In [ ]:
_MF_CELL=10
# UsageLogs는 실제 컨테이너의 created_time, terminated_time이 있음.
# 하지만 TimeSeries는 1분 단위의 로그이므로 UserTime을 계산하기에는 다소 오차가 있음.
# 이에, UsageLogs의 UsedTime을 갖다 쓰기로 함

df = df_3.copy()

# (가능하면) 0.x GPU 제외하고 싶으면 여기서 필터
# df = df[df["gpu_allocated(group)"] != "0.x GPU"].copy()

# Non-GPU 판정(벡터화)
is_cpu = df["gpu_allocated(group)"].astype(str).str.strip().eq("0 GPU")

# idle_flag (분 단위)
idle_cpu = is_cpu & (df["CPU"] <= CPU_IDLE_0GPU) & (df["Memory"] <= MEM_IDLE_0GPU)
idle_gpu = (~is_cpu) & (df["CPU"] <= CPU_IDLE_GPU) & (df["Memory"] <= MEM_IDLE_GPU) & (df["GPU"] <= GPU_IDLE_GPU)

df["idle_flag"] = (idle_cpu | idle_gpu)
df["state"] = np.where(df["idle_flag"], "idle", "used")

# filename 단위: used가 1번이라도 있으면 has_used=True
has_used = (
    (df["state"] == "used")
    .groupby(df["filename"])
    .any()
    .rename("has_used")
)

# peak 기반 라벨(당신 정의)
is_lowutil_new = (~has_used).map({True: "low_util", False: "normal"}).rename("is_lowutil_container")


# 1) (그대로) TimeSeries -> filename 단위 요약
grouped_df_3 = (
    df_3.groupby("filename", as_index=False)
        .agg({
            "used_time(minute)": "max",
            "CPU": "mean",
            "Memory": "mean",
            "GPU": "mean",
            "gpu_allocated(group)": "last",
        })
).rename(columns={"used_time(minute)": "used_time_ts(minute)"})

# ✅ has_used / is_lowutil_container 붙이기
grouped_df_3 = (
    grouped_df_3
      .set_index("filename")
      .join(has_used, how="left")
      .join(is_lowutil_new, how="left")
      .reset_index()
)

# 2) join_key는 "요약된 grouped_df_3"에서만 만든다 (✅ 여기서부터 빨라짐)
def extract_id_from_filename(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
         .str.replace("\\", "/", regex=False)
         .str.split("/").str[-1]
         .str.replace(".csv", "", regex=False)
         .str.strip()
         .str.split("-").str[-1]
    )

grouped_df_3["join_key"] = extract_id_from_filename(grouped_df_3["filename"])


# 3) UsageLog 쪽도 join_key 만들기 (UsageLog는 3만행 수준이라 부담 적음)
df_UsageLogJoin = df_UsageLog.copy()
df_UsageLogJoin["join_key"] = extract_id_from_filename(df_UsageLogJoin["filename"])

usage_rt_by_id = (
    df_UsageLogJoin.groupby("join_key", as_index=False)["used_time(minute)"]
              .max()
              .rename(columns={"used_time(minute)": "used_time_usage(minute)"})
)

# 4) merge + 덮어쓰기
merged = grouped_df_3.merge(usage_rt_by_id, on="join_key", how="left")
print("[DIAG] match rate:", merged["used_time_usage(minute)"].notna().mean())

merged["used_time(minute)"] = merged["used_time_usage(minute)"].fillna(merged["used_time_ts(minute)"])

# 5) 정리: 키/중간 컬럼 삭제
grouped_df_3 = merged.drop(columns=["join_key", "used_time_ts(minute)", "used_time_usage(minute)"])

print("[CHECK] done. rows:", len(grouped_df_3))

grouped_df_3 = grouped_df_3.sort_values(by=["gpu_allocated(group)"], inplace=False)


In [ ]:
_mf_capture_cell(10)


In [ ]:
_MF_CELL=11
# 정말 덮어써진 used_time이 들어갔는지 검증([DIAG] max abs diff: 0.0 이면 덮어쓰기 완료)
# (필수) grouped_df_3 used_time 분포 확인
print(grouped_df_3["used_time(minute)"].describe(percentiles=[0.01,0.1,0.5,0.9,0.99]))

# (필수) UsageLog와 조인해서 차이가 0에 가깝게 나오는지 확인
tmp = grouped_df_3[["filename","used_time(minute)"]].copy()
tmp["join_key"] = tmp["filename"].astype(str).str.replace(".csv","",regex=False).str.split("-").str[-1]

u = df_UsageLogJoin.copy()
u["join_key"] = u["filename"].astype(str).str.replace(".csv","",regex=False).str.split("-").str[-1]
u_rt = u.groupby("join_key", as_index=False)["used_time(minute)"].max().rename(columns={"used_time(minute)":"usage_rt"})

tmp = tmp.merge(u_rt, on="join_key", how="left")
d = (tmp["used_time(minute)"] - tmp["usage_rt"])
print("[DIAG] max abs diff:", d.abs().max())
print(d.describe(percentiles=[0.01,0.1,0.5,0.9,0.99]))


In [ ]:
_mf_capture_cell(11)


In [ ]:
_MF_CELL=12
# # 'is_lowutil_container' 컬럼에 low_util 기준을 충족하는 컨테이너는 low_util, 아니면 normal로 저장
# is_cpu_container = grouped_df_3["gpu_allocated(group)"].astype(str).eq("0 GPU")

# is_low_container = np.where(
#     is_cpu_container,
#     (grouped_df_3["CPU"] <= CPU_IDLE_0GPU) & (grouped_df_3["Memory"] <= MEM_IDLE_0GPU),
#     (grouped_df_3["CPU"] <= CPU_IDLE_GPU) & (grouped_df_3["Memory"] <= MEM_IDLE_GPU) & (grouped_df_3["GPU"] <= GPU_IDLE_GPU),
# )

# grouped_df_3["is_lowutil_container"] = np.where(is_low_container, "low_util", "normal")


# grouped_df_3 = grouped_df_3.sort_values(by=["gpu_allocated(group)"], inplace=False)


In [ ]:
_mf_capture_cell(12)


# TABLE 7
## DISTRIBUTION OF LOW-UTILIZATION CONTAINERS BY GPU ALLOCATION SIZE.


In [ ]:
_MF_CELL=14
# 1. count 테이블 (컨테이너 개수: filename 단위로 집계하는 게 더 정확)

# ============================================
# 0) 안전장치: 라벨 표준화 함수
# ============================================
def norm_lowutil_label(x):
    s = str(x).strip().lower()
    if s in ["low_util", "low-util", "lowutil", "low_utilization", "low-utilization", "low utilization"]:
        return "low_util"
    if s in ["normal", "norm"]:
        return "normal"
    return np.nan  # 알 수 없는 값은 NaN 처리

def norm_gpu_group(x):
    s = str(x).strip()
    # "0.0_GPU" 같은 포맷 방어
    if s.endswith("_GPU"):
        num = s.replace("_GPU", "")
        try:
            n = int(float(num))
            return f"{n} GPU"
        except:
            return np.nan
    # 이미 "0 GPU" 형태면 그대로
    if "GPU" in s:
        return s.replace(".0", "").replace("_", " ").strip()
    return np.nan

ORDER = ["0 GPU", "1 GPU", "2 GPU", "3 GPU", "4 GPU"]

# ============================================
# 1) 컨테이너 기준 테이블 (grouped_df_3: filename 단위 요약을 가정)
# ============================================
g = grouped_df_3.copy()

# 표준화
g["gpu_group_norm"] = g["gpu_allocated(group)"].apply(norm_gpu_group)
g["is_lowutil_norm"] = g["is_lowutil_container"].apply(norm_lowutil_label)

# 필요한 값만 유지
g = g.dropna(subset=["gpu_group_norm", "is_lowutil_norm"]).copy()

# count
cnt = (
    g.groupby("gpu_group_norm")["is_lowutil_norm"]
     .value_counts()
     .rename("count")
     .reset_index()
)

# pivot
tbl = (
    cnt.pivot(index="gpu_group_norm", columns="is_lowutil_norm", values="count")
      .fillna(0)
      .astype(int)
)

# (핵심) 없는 컬럼이 있어도 항상 생성되게
for col in ["low_util", "normal"]:
    if col not in tbl.columns:
        tbl[col] = 0

# 컬럼명 보기 좋게
tbl = tbl.rename(columns={
    "low_util": "Low-util containers",
    "normal": "Normal containers"
})

# Total, Ratio, Share
tbl["Total containers"] = tbl["Low-util containers"] + tbl["Normal containers"]

den = tbl["Total containers"].replace(0, np.nan)
tbl["Ratio of low-util (%)"] = (tbl["Low-util containers"] / den * 100).round(2).fillna(0)

tbl["Share of containers by GPU group (%)"] = (
    tbl["Total containers"] / tbl["Total containers"].sum() * 100
).round(2)

# order 정렬 + 없는 group도 표시하고 싶으면 reindex
tbl = tbl.reindex(ORDER).fillna(0)

# Total 행
total = pd.DataFrame({
    "Low-util containers": [int(tbl["Low-util containers"].sum())],
    "Normal containers":   [int(tbl["Normal containers"].sum())],
})
total["Total containers"] = total["Low-util containers"] + total["Normal containers"]
total["Ratio of low-util (%)"] = (
    total["Low-util containers"] / total["Total containers"].replace(0, np.nan) * 100
).round(2).fillna(0)
total["Share of containers by GPU group (%)"] = 100.00
total.index = ["Total"]

final_table = pd.concat([tbl, total], axis=0)

# 보기: name 제거(밀림 방지) + 컬럼 순서 고정
final_table.columns.name = None
final_table.index.name = None
final_table = final_table[[
    "Low-util containers",
    "Normal containers",
    "Total containers",
    "Ratio of low-util (%)",
    "Share of containers by GPU group (%)",
]]


final_table



In [ ]:
_mf_capture_cell(14)


# 0403 Low utilization 컨테이너와 Normal 컨테이너의 사용시간 분포 그래프
## 0403_low_vs_normal_container_usage_time_distribution


In [ ]:
_MF_CELL=16
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick

plt.figure(figsize=(12, 6))

order = ['0 GPU', '1 GPU', '2 GPU', '3 GPU', '4 GPU']
labels = [x.replace(' GPU', '') for x in order]

# 0 이하 값 방지
data = grouped_df_3.copy()
data['used_time(minute)'] = data['used_time(minute)'].clip(lower=1.0)

ax = sns.violinplot(
    x=data['gpu_allocated(group)'],
    y=data['used_time(minute)'],
    hue=data['is_lowutil_container'],
    order=order,
    log_scale=(False, True),
    inner='box',
    cut=1.5,
    bw_adjust=0.6
)

# 축 레이블
ax.set_ylabel('Run Time (minutes)', fontsize=25)
ax.set_xlabel('GPU Count Per Container', fontsize=25)
ax.set_xticklabels(labels, rotation=0, fontsize=25)
ax.tick_params(axis='y', labelsize=25)

# y축 로그 표기
ax.yaxis.set_major_locator(mtick.LogLocator(base=10))
ax.yaxis.set_minor_locator(mtick.LogLocator(base=10, subs=range(2, 10)))
ax.yaxis.set_major_formatter(mtick.LogFormatterMathtext(base=10))

# y축 범위
ax.set_ylim(1e-2, 1e8)

# 범례 (가로로 2개)
handles, _ = ax.get_legend_handles_labels()
plt.legend(
    handles,
    ['Normal', 'Low-Utilization'],
    title='',
    fontsize=25,
    loc='upper right',
    ncol=2
)

plt.tight_layout()
plt.show()


In [ ]:
_mf_capture_cell(16)


In [ ]:
_MF_CELL=17
summary = (data
  .groupby(['gpu_allocated(group)', 'is_lowutil_container'])['used_time(minute)']
  .agg(n='count',
       p50=lambda x: x.quantile(0.5),
       p90=lambda x: x.quantile(0.9),
       p99=lambda x: x.quantile(0.99),
       mean='mean')
  .reset_index()
)
print(summary.sort_values(['gpu_allocated(group)', 'is_lowutil_container']))


In [ ]:
_mf_capture_cell(17)


In [ ]:
_MF_CELL=18
# =========================================================
# (0) 입력: df_3, 임계값 변수들이 이미 위에서 정의되어 있어야 함
# =========================================================
req_thr = ["CPU_IDLE_0GPU", "MEM_IDLE_0GPU", "CPU_IDLE_GPU", "MEM_IDLE_GPU", "GPU_IDLE_GPU"]
miss_thr = [v for v in req_thr if v not in globals()]
if miss_thr:
    raise NameError(f"다음 임계값 변수가 없습니다: {miss_thr}")

# =========================================================
# 1) 입력 DF 준비
# =========================================================
df = df_3.copy()

need_cols = ["filename", "time", "used_time(minute)", "CPU", "Memory", "GPU", "gpu_allocated(group)"]
miss = [c for c in need_cols if c not in df.columns]
if miss:
    raise KeyError(f"df_3에 다음 컬럼이 없습니다: {miss}")

df = df.dropna(subset=["filename", "time"]).copy()
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["time"]).copy()

df["used_time(minute)"] = pd.to_numeric(df["used_time(minute)"], errors="coerce")
df = df.dropna(subset=["used_time(minute)"]).copy()

df["filename"] = df["filename"].astype(str).str.strip()

# 선택: 메모리/속도 개선 (대용량일 때 체감됨)
df["filename"] = df["filename"].astype("category")
df["gpu_allocated(group)"] = df["gpu_allocated(group)"].astype(str).str.strip()

# =========================================================
# 2) 1분 샘플링 전제: 중복 샘플 제거
# =========================================================
df = df.sort_values(["filename", "used_time(minute)", "time"]).copy()
df = df.drop_duplicates(subset=["filename", "used_time(minute)"], keep="last").copy()

# =========================================================
# 3) Non-GPU/GPU-Enabled 판정(안정화)
# =========================================================
def _nongpu_from_group_str(s: pd.Series) -> pd.Series:
    g = s.astype(str).str.strip()

    is0 = g.isin(["0 GPU", "0.0_GPU", "0_GPU"])
    is0 = is0 | g.str.contains(r"\b0\s*GPU\b", regex=True)

    num = g.str.extract(r"^\s*([0-9]*\.?[0-9]+)\s*_GPU\s*$", expand=False)
    num = pd.to_numeric(num, errors="coerce")
    is0 = is0 | (num == 0.0)

    return is0.fillna(False)

if "gpu_allocated(EA)" in df.columns:
    ea = pd.to_numeric(df["gpu_allocated(EA)"], errors="coerce")
    is_cpu = (ea == 0.0) & ea.notna()
    need_fallback = ea.isna()
    if need_fallback.any():
        is_cpu.loc[need_fallback] = _nongpu_from_group_str(df.loc[need_fallback, "gpu_allocated(group)"])
else:
    is_cpu = _nongpu_from_group_str(df["gpu_allocated(group)"])

# =========================================================
# 4) idle/used 판정
# =========================================================
df["CPU"] = pd.to_numeric(df["CPU"], errors="coerce")
df["Memory"] = pd.to_numeric(df["Memory"], errors="coerce")
df["GPU"] = pd.to_numeric(df["GPU"], errors="coerce")

# 숫자 변환 실패 제거 (비교 안정화)
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["CPU", "Memory"]).copy()
df = df.loc[is_cpu.loc[df.index] | df["GPU"].notna()].copy()

# 인덱스 정렬/맞춤 보장
is_cpu = is_cpu.loc[df.index]

idle_cpu = is_cpu & (df["CPU"] <= CPU_IDLE_0GPU) & (df["Memory"] <= MEM_IDLE_0GPU)
idle_gpu = (~is_cpu) & (df["CPU"] <= CPU_IDLE_GPU) & (df["Memory"] <= MEM_IDLE_GPU) & (df["GPU"] <= GPU_IDLE_GPU)

df["idle_flag"] = (idle_cpu | idle_gpu).astype(np.int8)

# state 문자열은 꼭 필요할 때만 만드는 게 좋음 (성능)
# df["state"] = np.where(df["idle_flag"] == 1, "idle", "used")

# =========================================================
# 5) 파일 단위 집계 (groupby 재사용)
# =========================================================
g = df.groupby("filename", observed=True)

runtime = g["used_time(minute)"].nunique().rename("runtime_minutes").astype(float)
idle_minutes = g["idle_flag"].sum().rename("idle_minutes").astype(float)
idle_ratio = (idle_minutes / runtime.replace(0, np.nan)).fillna(0.0).rename("idle_ratio")

# has_used: 문자열 state 대신 idle_flag 기반으로 계산 (빠름)
# idle_flag가 전부 1이면 Low-Utilization, 하나라도 0이면 Normal
has_used = (g["idle_flag"].min() == 0).rename("has_used")

# meta
meta = g["gpu_allocated(group)"].last().rename("gpu_allocated(group)")

# =========================================================
# 6) first/last used 기반 지표 (분 단위)
#    - used 샘플만 필터 후 groupby (여기도 state 대신 idle_flag 사용)
# =========================================================
df_used = df.loc[df["idle_flag"] == 0, ["filename", "used_time(minute)"]].copy()
g_used = df_used.groupby("filename", observed=True)

first_used_time = g_used["used_time(minute)"].min().rename("first_used_time")
last_used_time  = g_used["used_time(minute)"].max().rename("last_used_time")

idle_before_first_used = (first_used_time - 1).clip(lower=0.0).rename("idle_before_first_used")
idle_after_last_used = (runtime - last_used_time).clip(lower=0.0).rename("idle_after_last_used")

# =========================================================
# 7) meta 붙이기 (filename 단위) - Normal/Low-Utilization
# =========================================================
is_lowutil_new = (~has_used).map({True: "Low-Utilization", False: "Normal"}).rename("is_lowutil_new")

df_file_idle = (
    pd.concat([runtime, idle_minutes, idle_ratio, idle_before_first_used, idle_after_last_used], axis=1)
      .join(meta, how="left")
      .join(is_lowutil_new, how="left")
      .reset_index()
)

# category였으면 보기 좋게 문자열로 복원 (선택)
df_file_idle["filename"] = df_file_idle["filename"].astype(str)

# =========================================================
# 8) 진단
# =========================================================
print("[DIAG] df_file_idle rows:", len(df_file_idle))
print("[DIAG] idle_ratio min/max:", df_file_idle["idle_ratio"].min(), df_file_idle["idle_ratio"].max())

# =========================================================
# 9) High-idle 비율 요약 (gpu_allocated(group)별)
# =========================================================
IDLE_RATIO_HIGH = 1.00  # 100%

# Normal only 필터 (문구 일치)
df_norm = df_file_idle.loc[df_file_idle["is_lowutil_new"] == "Normal"].copy()

high_share = (
    df_norm.assign(
        is_high=np.isclose(df_norm["idle_ratio"], 1.0, rtol=0, atol=1e-12)
    )
    .groupby("gpu_allocated(group)", dropna=False)["is_high"]
    .mean()
    .mul(100)
    .round(2)
    .sort_values(ascending=False)
)

print("[Normal only] Idle ratio == 100% proportion by gpu_allocated(group) (%):")
print(high_share)

# (추천) 분모/분자 같이 보기
high_count_tbl = (
    df_norm.assign(
        is_high=np.isclose(df_norm["idle_ratio"], 1.0, rtol=0, atol=1e-12)
    )
    .groupby("gpu_allocated(group)", dropna=False)["is_high"]
    .agg(total="size", high_100pct="sum")
)
high_count_tbl["share_%"] = (high_count_tbl["high_100pct"] / high_count_tbl["total"] * 100).round(2)

print("\n[Normal only] Count summary by gpu_allocated(group):")
print(high_count_tbl.sort_values("total", ascending=False))


In [ ]:
_mf_capture_cell(18)


In [ ]:
_MF_CELL=19



In [ ]:
_mf_capture_cell(19)


# Figure 9
## 바이올린 플롯: idle_ratio
## 0403_idle_ratio_of_normal_containers


In [ ]:
_MF_CELL=21
# =========================================================
# [Standalone] Violin plot: (Normal only) idle_ratio
# - 입력 의존성 최소화: df_3 + 임계값 변수들만 필요
# - 내부에서 file-level 요약(df_file_idle 역할)까지 생성
# =========================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---------------------------------------------------------
# 0) 필수 입력/임계값 체크
# ---------------------------------------------------------
req_thr = ["CPU_IDLE_0GPU", "MEM_IDLE_0GPU", "CPU_IDLE_GPU", "MEM_IDLE_GPU", "GPU_IDLE_GPU"]
miss_thr = [v for v in req_thr if v not in globals()]
if miss_thr:
    raise NameError(f"다음 임계값 변수가 없습니다: {miss_thr}")

if "df_3" not in globals():
    raise NameError("df_3가 정의되어 있어야 합니다.")

# ---------------------------------------------------------
# 1) 입력 DF 준비 (원본 셀과 동일한 흐름)
# ---------------------------------------------------------
df = df_3.copy()

need_cols = ["filename", "time", "used_time(minute)", "CPU", "Memory", "GPU", "gpu_allocated(group)"]
miss = [c for c in need_cols if c not in df.columns]
if miss:
    raise KeyError(f"df_3에 다음 컬럼이 없습니다: {miss}")

df = df.dropna(subset=["filename", "time"]).copy()
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["time"]).copy()

df["used_time(minute)"] = pd.to_numeric(df["used_time(minute)"], errors="coerce")
df = df.dropna(subset=["used_time(minute)"]).copy()

df["filename"] = df["filename"].astype(str).str.strip()

# ---------------------------------------------------------
# 2) 1분 샘플링 전제: 중복 샘플 제거
# ---------------------------------------------------------
df = df.sort_values(["filename", "used_time(minute)", "time"]).copy()
df = df.drop_duplicates(subset=["filename", "used_time(minute)"], keep="last").copy()

# ---------------------------------------------------------
# 3) Non-GPU/GPU-Enabled 판정 (vectorized + fallback)
# ---------------------------------------------------------
def _nongpu_from_group_str(s: pd.Series) -> pd.Series:
    g = s.astype(str).str.strip()

    # 흔한 표기들
    is0 = g.isin(["0 GPU", "0_GPU", "0.0_GPU"])

    # "0 GPU" 형태(공백 변형 포함)
    is0 = is0 | g.str.contains(r"^\s*0(?:\.0+)?\s*GPU\s*$", regex=True)

    # "0.0_GPU", "1_GPU" 형태 숫자 파싱
    num = g.str.extract(r"^\s*([0-9]*\.?[0-9]+)\s*_GPU\s*$", expand=False)
    num = pd.to_numeric(num, errors="coerce")
    is0 = is0 | (num == 0.0)

    return is0.fillna(False)

if "gpu_allocated(EA)" in df.columns:
    ea = pd.to_numeric(df["gpu_allocated(EA)"], errors="coerce")
    is_cpu = (ea == 0.0) & ea.notna()

    need_fallback = ea.isna()
    if need_fallback.any():
        is_cpu.loc[need_fallback] = _nongpu_from_group_str(df.loc[need_fallback, "gpu_allocated(group)"])
else:
    is_cpu = _nongpu_from_group_str(df["gpu_allocated(group)"])

# ---------------------------------------------------------
# 4) idle/used 판정
# ---------------------------------------------------------
df["CPU"] = pd.to_numeric(df["CPU"], errors="coerce")
df["Memory"] = pd.to_numeric(df["Memory"], errors="coerce")
df["GPU"] = pd.to_numeric(df["GPU"], errors="coerce")

# 숫자 변환 실패 행 제거 (비교 연산 안정화)
df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=["CPU", "Memory"]).copy()
df = df.loc[is_cpu.loc[df.index] | df["GPU"].notna()].copy()

idle_cpu = is_cpu.loc[df.index] & (df["CPU"] <= CPU_IDLE_0GPU) & (df["Memory"] <= MEM_IDLE_0GPU)
idle_gpu = (~is_cpu.loc[df.index]) & (df["CPU"] <= CPU_IDLE_GPU) & (df["Memory"] <= MEM_IDLE_GPU) & (df["GPU"] <= GPU_IDLE_GPU)

df["idle_flag"] = (idle_cpu | idle_gpu).astype(np.int8)
df["state"] = np.where(df["idle_flag"] == 1, "idle", "used")

# ---------------------------------------------------------
# 5) 1분 주기 샘플링 기반 적분 (file-level 요약 생성)
# ---------------------------------------------------------
runtime = df.groupby("filename")["used_time(minute)"].nunique().rename("runtime_minutes").astype(float)
idle_minutes = df.groupby("filename")["idle_flag"].sum().rename("idle_minutes").astype(float)
idle_ratio = (idle_minutes / runtime.replace(0, np.nan)).fillna(0.0).rename("idle_ratio")

# first/last used 기반 지표
first_used_time = (
    df.loc[df["state"] == "used"]
      .groupby("filename")["used_time(minute)"]
      .min()
      .rename("first_used_time")
)
last_used_time = (
    df.loc[df["state"] == "used"]
      .groupby("filename")["used_time(minute)"]
      .max()
      .rename("last_used_time")
)

idle_before_first_used = (first_used_time - 1).clip(lower=0.0).rename("idle_before_first_used")
idle_after_last_used = (runtime - last_used_time).clip(lower=0.0).rename("idle_after_last_used")

# meta + normal/lowutil
meta_group = (
    df.groupby("filename")["gpu_allocated(group)"]
      .last()
      .rename("gpu_allocated(group)")
)

has_used = (
    (df["state"] == "used")
    .groupby(df["filename"])
    .any()
    .rename("has_used")
)

is_lowutil_new = (~has_used).map({True: "Low-Utilization", False: "Normal"}).rename("is_lowutil_new")

# (중요) 여기서 df_file_idle를 내부 생성하지만, 외부 의존 없이 셀 내에서 끝냄
df_file_idle_local = (
    pd.concat(
        [runtime, idle_minutes, idle_ratio, idle_before_first_used, idle_after_last_used],
        axis=1
    )
    .join(meta_group, how="left")
    .join(is_lowutil_new, how="left")
    .reset_index()
)

print("[DIAG] df_file_idle_local rows:", len(df_file_idle_local))
print("[DIAG] idle_ratio min/max:",
      df_file_idle_local["idle_ratio"].min(),
      df_file_idle_local["idle_ratio"].max())

# ---------------------------------------------------------
# 6) Violin plot용 plot_df 생성 (Normal only, Non-GPU vs GPU-Enabled)
# ---------------------------------------------------------
plot_df = df_file_idle_local.copy()

# gpu_allocated(group) -> gpu_bin (안전 파싱)
plot_df["gpu_bin"] = np.where(
    _nongpu_from_group_str(plot_df["gpu_allocated(group)"]),
    "Non-GPU",
    "GPU-Enabled"
)

# Normal only
plot_df = plot_df.loc[plot_df["is_lowutil_new"] == "Normal"].copy()

# 안전 처리
plot_df["idle_ratio"] = pd.to_numeric(plot_df["idle_ratio"], errors="coerce")
plot_df = (
    plot_df.replace([np.inf, -np.inf], np.nan)
           .dropna(subset=["idle_ratio", "gpu_bin"])
           .copy()
)

# 범위 보정 (이론상 0~1이어야 함)
plot_df["idle_ratio"] = plot_df["idle_ratio"].clip(lower=0.0, upper=1.0)

# 플롯용 percent 변환
plot_df["idle_ratio_pct"] = plot_df["idle_ratio"] * 100.0

order2 = ["Non-GPU", "GPU-Enabled"]

# ---------------------------------------------------------
# 7) Violin plot
# ---------------------------------------------------------
plt.figure(figsize=(15, 8))
ax = sns.violinplot(
    data=plot_df,
    x="gpu_bin",
    y="idle_ratio_pct",
    order=order2,
    inner="box",
    cut=2,
    bw_adjust=0.6,
    scale="width"
)

ax.set_title("Idle Ratio Distribution (Normal Containers)", fontsize=35)
ax.set_xlabel("Device Type", fontsize=35)
ax.set_ylabel("Idle Ratio (%)", fontsize=35)

ax.tick_params(axis="x", labelsize=35)
ax.tick_params(axis="y", labelsize=35)

ax.set_ylim(-10, 110)
ax.grid(False)

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# 8) Idle ratio == 100% 비율 요약 (gpu_bin 기준)
# ---------------------------------------------------------
high_share = (
    plot_df.assign(
        is_high=np.isclose(plot_df["idle_ratio"], 1.0, rtol=0, atol=1e-12)
    )
    .groupby("gpu_bin", dropna=False)["is_high"]
    .mean()
    .mul(100)
    .round(2)
)

print("[Normal only] Idle ratio == 100% proportion by gpu_bin (%):")
print(high_share.reindex(order2).fillna(0.0))

# (선택) 분모/분자 같이 보기
high_count_tbl = (
    plot_df.assign(
        is_high=np.isclose(plot_df["idle_ratio"], 1.0, rtol=0, atol=1e-12)
    )
    .groupby("gpu_bin", dropna=False)["is_high"]
    .agg(total="size", high_100pct="sum")
)

high_count_tbl["share_%"] = (high_count_tbl["high_100pct"] / high_count_tbl["total"] * 100).round(2)

print("\n[Normal only] Count summary by gpu_bin:")
print(high_count_tbl.reindex(order2).fillna(0))


In [ ]:
_mf_capture_cell(21)


In [ ]:
_MF_CELL=22



In [ ]:
_mf_capture_cell(22)


In [ ]:
_MF_CELL=23



In [ ]:
_mf_capture_cell(23)


In [ ]:
_MF_CELL=24
# 위의 Runtime을 비교하는것보다, 본 셀의 Idle 시간 Ratio가 두 컨테이너를 잘 구분하므로, 이걸 쓰는게 좋아보임


# =========================================================
# 1) 입력 DF 준비
# =========================================================
df = df_3.copy()

# 필수 컬럼 체크
need_cols = ["filename", "time", "used_time(minute)", "CPU", "Memory", "GPU", "gpu_allocated(group)"]
miss = [c for c in need_cols if c not in df.columns]
if miss:
    raise KeyError(f"df_3에 다음 컬럼이 없습니다: {miss}")

df = df.dropna(subset=["filename", "time"]).copy()
df["time"] = pd.to_datetime(df["time"], errors="coerce")
df = df.dropna(subset=["time"]).copy()

df["used_time(minute)"] = pd.to_numeric(df["used_time(minute)"], errors="coerce")
df = df.dropna(subset=["used_time(minute)"]).copy()

df["filename"] = df["filename"].astype(str).str.strip()

# =========================================================
# 2) (중요) 1분 샘플링 전제: 중복 샘플 제거
#    - 같은 filename에서 같은 used_time(minute)가 여러 번 나오면
#      1분을 중복 합산하지 않도록 1개만 남김
# =========================================================
df = df.sort_values(["filename", "used_time(minute)", "time"]).copy()
df = df.drop_duplicates(subset=["filename", "used_time(minute)"], keep="last").copy()

# =========================================================
# 3) Non-GPU/GPU-Enabled 판정 (안정화)
#    - gpu_allocated(EA)가 있으면 그걸 최우선
#    - 없으면 gpu_allocated(group) 문자열 파싱
# =========================================================
def _is_nongpu_row(r):
    # 1) EA 우선
    if "gpu_allocated(EA)" in r.index:
        try:
            ea = float(r["gpu_allocated(EA)"])
            if not np.isnan(ea):
                return ea == 0.0
        except:
            pass

    # 2) group fallback
    g = str(r["gpu_allocated(group)"]).strip()
    # 흔한 표기들 대응
    if g in ["0 GPU", "0.0_GPU", "0_GPU"]:
        return True
    # '0.0_GPU' 같은 패턴
    if g.endswith("_GPU"):
        try:
            v = float(g.replace("_GPU", ""))
            return v == 0.0
        except:
            return False
    if "0 GPU" in g:
        return True
    return False

# Equivalent vectorized classification; verify against original row function.
assert df['gpu_allocated(group)'].isin(['0 GPU','1 GPU','2 GPU','3 GPU','4 GPU']).all()
is_cpu = df['gpu_allocated(group)'].eq('0 GPU')
if 'gpu_allocated(EA)' in df.columns:
    _ea=pd.to_numeric(df['gpu_allocated(EA)'],errors='coerce')
    is_cpu=is_cpu.where(_ea.isna(),_ea.eq(0))
_sample = df.iloc[::max(1,len(df)//1000)]
assert np.array_equal(is_cpu.loc[_sample.index].to_numpy(),_sample.apply(_is_nongpu_row,axis=1).to_numpy())

# =========================================================
# 4) idle/used 판정
# =========================================================
idle_cpu = is_cpu & (df["CPU"] <= CPU_IDLE_0GPU) & (df["Memory"] <= MEM_IDLE_0GPU)
idle_gpu = (~is_cpu) & (df["CPU"] <= CPU_IDLE_GPU) & (df["Memory"] <= MEM_IDLE_GPU) & (df["GPU"] <= GPU_IDLE_GPU)

df["idle_flag"] = (idle_cpu | idle_gpu).astype(np.int8)
df["state"] = np.where(df["idle_flag"] == 1, "idle", "used")

# =========================================================
# 5) 1분 주기 샘플링 기반 적분
#    - dt_min은 "항상 1분"
#    - runtime_minutes = 샘플 개수 (혹은 max-min+1)
#    - idle_minutes = idle 샘플 개수
# =========================================================
df["dt_min"] = 1.0  # 1분 샘플링

# runtime: 관측된 분(minute)의 개수
runtime = df.groupby("filename")["used_time(minute)"].nunique().rename("runtime_minutes").astype(float)

# idle_minutes: idle_flag==1인 분의 개수
idle_minutes = (df["idle_flag"].groupby(df["filename"]).sum()).rename("idle_minutes").astype(float)

idle_ratio = (idle_minutes / runtime.replace(0, np.nan)).fillna(0.0).rename("idle_ratio")

# =========================================================
# 6) first/last used 기반 지표 (분 단위)
# =========================================================
first_used_time = (
    df.loc[df["state"] == "used"]
      .groupby("filename")["used_time(minute)"]
      .min()
      .rename("first_used_time")
)

last_used_time = (
    df.loc[df["state"] == "used"]
      .groupby("filename")["used_time(minute)"]
      .max()
      .rename("last_used_time")
)

# used가 아예 없는 컨테이너는 first/last가 NaN -> 의미상 runtime 전체 idle로 처리
idle_before_first_used = (first_used_time - 1).clip(lower=0.0).rename("idle_before_first_used")
idle_after_last_used = (runtime - last_used_time).clip(lower=0.0).rename("idle_after_last_used")

# =========================================================
# 7) meta 붙이기 (filename 단위)
#    - has_used: 한번이라도 used 되었다면, normal 컨테이너로 변경
# =========================================================
# filename 단위로 GPU group만 붙이기
meta = (
    df.groupby("filename")["gpu_allocated(group)"]
      .last()
      .rename("gpu_allocated(group)")
)

has_used = (
    (df["state"] == "used")
    .groupby(df["filename"])
    .any()
    .rename("has_used")
)

# "used가 1번이라도 있으면 Normal" / 아니면 Low-Utilization
is_lowutil_new = (~has_used).map({True: "Low-Utilization", False: "Normal"}).rename("is_lowutil_new")

df_file_idle = (
    pd.concat([runtime, idle_minutes, idle_ratio, idle_before_first_used, idle_after_last_used], axis=1)
      .join(meta, how="left")
      .join(is_lowutil_new, how="left")
      .reset_index()
)

# =========================================================
# 8) 진단 (idle_ratio 범위 확인)
# =========================================================
print("[DIAG] df_file_idle rows:", len(df_file_idle))
print(df_file_idle[["runtime_minutes", "idle_minutes", "idle_ratio",
                    "idle_before_first_used", "idle_after_last_used"]]
      .describe(percentiles=[.1, .5, .9, .99]))

print("[DIAG] idle_ratio min/max:", df_file_idle["idle_ratio"].min(), df_file_idle["idle_ratio"].max())

# =========================================================
# 9) 바이올린 플롯: (Normal only) idle_ratio
#    - Peak 기반 정의에서는 Low-Util(used 없음) => idle_ratio=1.0이 당연
#    - 따라서 Normal 내부에서 idle_ratio가 높게 나타나는 비율을 강조
# =========================================================
order = ['0 GPU', '1 GPU', '2 GPU', '3 GPU', '4 GPU']
labels = [x.replace(' GPU', '') for x in order]

plot_df = df_file_idle.copy()
plot_df = plot_df[plot_df["gpu_allocated(group)"].isin(order)].copy()

# ✅ Peak 기반 라벨 기준으로 Normal만 남김
# (주의: df_file_idle에 is_lowutil_new는 "Normal"/"Low-Utilization" 문자열)
plot_df = plot_df[plot_df["is_lowutil_new"] == "Normal"].copy()

# (선택) 혹시라도 runtime 0/NaN 방지
plot_df = plot_df.replace([np.inf, -np.inf], np.nan).dropna(subset=["idle_ratio"])

# ✅ 'idle_ratio가 높다'의 기준선 설정 (논문 메시지에 맞게 조정)
IDLE_RATIO_HIGH = 0.90   # 예: runtime의 90% 이상이 idle

plt.figure(figsize=(12, 6))
ax = sns.violinplot(
    data=plot_df,
    x="gpu_allocated(group)",
    y="idle_ratio",
    order=order,
    inner="box",
    cut=1.5,
    bw_adjust=0.6
)

ax.set_xlabel("GPU Count Per Container", fontsize=25)
ax.set_ylabel("Idle Ratio (idle/runtime)", fontsize=25)
ax.set_xticklabels(labels, fontsize=25)
ax.tick_params(axis='y', labelsize=25)


plt.tight_layout()
plt.show()

print(f"[Normal only] High-idle ratio >= {IDLE_RATIO_HIGH:.2f} proportion by GPU group:")



In [ ]:
_mf_capture_cell(24)


In [ ]:
_MF_CELL=25



In [ ]:
_mf_capture_cell(25)


In [ ]:
_MF_CELL=26



In [ ]:
_mf_capture_cell(26)


In [ ]:
_MF_CELL=27



In [ ]:
_mf_capture_cell(27)


In [ ]:
_MF_CELL=28
# Low utilization 컨테이너와 Normal 컨테이너의 사용시간 평균
grouped_normal = grouped_df_3[grouped_df_3['is_lowutil_container'] == 'normal'].groupby('gpu_allocated(group)')['used_time(minute)'].mean()
grouped_low_util = grouped_df_3[grouped_df_3['is_lowutil_container'] == 'low_util'].groupby('gpu_allocated(group)')['used_time(minute)'].mean()

# Merging both results into a single dataframe
grouped_df_3_by_gpu_combined = pd.DataFrame({
    'low_util Used time(minutes)': round(grouped_low_util, 2),
    'Normal Used time(minutes)': round(grouped_normal, 2)
    }).reset_index()

grouped_df_3_by_gpu_combined


In [ ]:
_mf_capture_cell(28)


In [ ]:
_MF_CELL=29
import numpy as np
import pandas as pd
import itertools
import time
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.ticker as mtick

# ============================================================
# [A] 그리드 후보(요구사항 반영)
#  - CPU/MEM: 0 제외, 5 단위
#  - GPU idle 조건: 0% 고정 (엄격)
# ============================================================
CPU_0GPU_CAND = [5, 10, 15, 20]
MEM_0GPU_CAND = [5, 10, 15, 20, 25]

CPU_GPU_CAND  = [5, 10, 15, 20]
MEM_GPU_CAND  = [5, 10, 15, 20, 25]

GPU_IDLE_FIXED = 0  # ✅ GPU 컨테이너 idle 판정은 GPU==0만 허용(엄격)
PCTS = [50, 90, 95]

print("grid cases =", len(CPU_0GPU_CAND)*len(MEM_0GPU_CAND)*len(CPU_GPU_CAND)*len(MEM_GPU_CAND))

# ============================================================
# [B] 데이터 준비 (한 번만)
#  - df_3 전체를 filename/time 정렬 후 numpy로 변환
#  - "Normal 컨테이너"는 grid마다 has_used(any used)로 filename 단위에서 결정
# ============================================================
base = df_3.dropna(subset=["filename", "time", "used_time(minute)"]).copy()
base["time"] = pd.to_datetime(base["time"], errors="coerce")
base = base.dropna(subset=["time"]).sort_values(["filename", "time"])

# filename을 정수 코드로 변환
file_codes, file_uniques = pd.factorize(base["filename"], sort=False)
n_files = len(file_uniques)

# numpy 배열
ut  = base["used_time(minute)"].to_numpy(dtype=np.float64)
cpu = base["CPU"].to_numpy(dtype=np.float64)
mem = base["Memory"].to_numpy(dtype=np.float64)
gpu = base["GPU"].to_numpy(dtype=np.float64)

# 0 GPU 컨테이너 여부 (row 기준)
is0 = (base["gpu_allocated(group)"].astype(str).to_numpy() == "0 GPU")

# runtime(컨테이너별) = used_time max
runtime = np.full(n_files, -np.inf, dtype=np.float64)
np.maximum.at(runtime, file_codes, ut)
runtime[runtime < 0] = np.nan

# 각 컨테이너 마지막 row 위치
pos = np.arange(len(file_codes), dtype=np.int64)
last_pos = np.full(n_files, -1, dtype=np.int64)
np.maximum.at(last_pos, file_codes, pos)

print("n_rows =", len(base), "/ n_files =", n_files, "/ last_pos valid ratio =", round(float(np.mean(last_pos>=0)), 4))

# ============================================================
# [C] Grid Search
#  - 핵심 목표: "USED가 있었다가, 마지막이 IDLE로 끝난" 컨테이너만 잡아서
#              tail idle(runtime - last_used_time)을 계산
# ============================================================
results = []
t0 = time.time()

# Equivalent computation: cache each of the 20 settings for each disjoint GPU bin.
# The original 400 combinations repeat these same per-bin calculations.
_starts=np.r_[0,np.flatnonzero(file_codes[1:] != file_codes[:-1])+1]
assert np.array_equal(file_codes[_starts],np.arange(n_files))
_cpu_settings={}
_gpu_settings={}
for _CPU0,_MEM0 in itertools.product(CPU_0GPU_CAND,MEM_0GPU_CAND):
    _idle=is0 & (cpu <= _CPU0) & (mem <= _MEM0)
    _used=is0 & ~_idle
    _last=np.maximum.reduceat(np.where(_used,ut,-np.inf),_starts)
    _cpu_settings[(_CPU0,_MEM0)]=(_last,_idle[last_pos])
for _CPUG,_MEMG in itertools.product(CPU_GPU_CAND,MEM_GPU_CAND):
    _idle=(~is0) & (cpu <= _CPUG) & (mem <= _MEMG) & (gpu <= GPU_IDLE_FIXED)
    _used=(~is0) & ~_idle
    _last=np.maximum.reduceat(np.where(_used,ut,-np.inf),_starts)
    _gpu_settings[(_CPUG,_MEMG)]=(_last,_idle[last_pos])
for CPU0,MEM0,CPUG,MEMG in itertools.product(CPU_0GPU_CAND,MEM_0GPU_CAND,CPU_GPU_CAND,MEM_GPU_CAND):
    _lc,_ic=_cpu_settings[(CPU0,MEM0)]
    _lg,_ig=_gpu_settings[(CPUG,MEMG)]
    last_used=np.maximum(_lc,_lg)
    has_used=last_used > -np.inf
    if not np.any(has_used):continue
    last_state_idle=_ic | _ig
    target=has_used & last_state_idle
    n_target=int(np.sum(target))
    if n_target == 0:continue
    tail_idle=np.clip(runtime[target]-last_used[target],0,None)
    p50,p90,p95=np.percentile(tail_idle,PCTS)
    results.append({'CPU0':CPU0,'MEM0':MEM0,'CPUG':CPUG,'MEMG':MEMG,
        'n_used_ever':int(np.sum(has_used)),'n_used_then_idle_end':n_target,
        'share_used_then_idle_end(%)':round(n_target/np.sum(has_used)*100,2),
        'tail_p50':round(float(p50),2),'tail_p90':round(float(p90),2),'tail_p95':round(float(p95),2),
        'n_normal':int(np.sum(has_used)),'n_low':int(n_files-np.sum(has_used))})

grid_df = pd.DataFrame(results)

print("elapsed(sec) =", round(time.time()-t0, 2), "/ rows =", len(grid_df))

# ------------------------------------------------------------
# "베스트" 기준:
#  1) used->idle_end 케이스 수가 많을수록 좋음(그림이 잘 나옴)
#  2) 같은 경우라면 tail_p90, tail_p95가 너무 이상치로 튀지 않는 조합 선호
# ------------------------------------------------------------
grid_df = grid_df.sort_values(
    by=["n_used_then_idle_end", "tail_p90", "tail_p95"],
    ascending=[False, True, True]
).reset_index(drop=True)

display(grid_df.head(10))  # 상위 10개 확인

# ============================================================
# [D] 상위 5개 조합으로 "tail idle" 그림 그리기
#  - (1) Violin(로그) : tail idle 분포
#  - (2) Percentile curve(로그 y) : CDF 스타일(percentile table을 곡선으로)
# ============================================================
topk = 5
top5 = grid_df.head(topk).copy()

# 어떤 조합을 그릴지 ID 문자열로 보기 쉽게 만듦
top5["combo"] = top5.apply(lambda r: f"CPU0={r.CPU0},MEM0={r.MEM0} | CPUG={r.CPUG},MEMG={r.MEMG},GPU=0", axis=1)
display(top5[["combo","n_used_then_idle_end","share_used_then_idle_end(%)","tail_p50","tail_p90","tail_p95"]])

# 공통: percentiles 축
percentiles = [50, 60, 70, 80, 90, 95]

for i, row in top5.iterrows():
    CPU0, MEM0, CPUG, MEMG = int(row["CPU0"]), int(row["MEM0"]), int(row["CPUG"]), int(row["MEMG"])
    combo_name = row["combo"]

    # ---------- (1) idle/used 마스크 재계산 ----------
    idle0 = is0 & (cpu <= CPU0) & (mem <= MEM0)
    idleg = (~is0) & (cpu <= CPUG) & (mem <= MEMG) & (gpu <= GPU_IDLE_FIXED)
    idle  = idle0 | idleg
    used_mask = ~idle

    # last_used_time per file
    last_used = np.full(n_files, -np.inf, dtype=np.float64)
    np.maximum.at(last_used, file_codes[used_mask], ut[used_mask])
    has_used = last_used > -np.inf

    # last row idle?
    last_state_idle = idle[last_pos]

    # target: USED 있었다가 마지막이 IDLE
    target = has_used & last_state_idle
    if np.sum(target) == 0:
        print("[SKIP] no target:", combo_name)
        continue

    # tail idle
    tail_idle = runtime[target] - last_used[target]
    tail_idle = np.clip(tail_idle, 0, None)


In [ ]:
_mf_capture_cell(29)


In [ ]:
_MF_CELL=30
# =========================================================
# Normal 컨테이너(=filename 단위 has_used True)만 추리기
#  - df_3["is_lowutil"] (row 단위) 사용 금지
#  - idle/used 판정은 threshold 기반
# =========================================================
base = (
    df_3.dropna(subset=["filename", "time", "used_time(minute)"])
       .copy()
)

base["time"] = pd.to_datetime(base["time"], errors="coerce")
base = base.dropna(subset=["time"]).copy()

# 필요하면 숫자형 안전 변환(권장)
base["used_time(minute)"] = pd.to_numeric(base["used_time(minute)"], errors="coerce")
base["CPU"]    = pd.to_numeric(base["CPU"], errors="coerce")
base["Memory"] = pd.to_numeric(base["Memory"], errors="coerce")
base["GPU"]    = pd.to_numeric(base["GPU"], errors="coerce")
base = base.dropna(subset=["used_time(minute)", "CPU", "Memory"]).copy()
base = base.loc[base["gpu_allocated(group)"].eq("0 GPU") | base["GPU"].notna()].copy()

# (선택) 0.x GPU 제외
# base = base[base["gpu_allocated(group)"].astype(str).str.strip().ne("0.x GPU")].copy()

# 0 GPU 판정 (벡터화)
is0 = base["gpu_allocated(group)"].astype(str).str.strip().eq("0 GPU")

# idle 판정(분 단위)
idle0 = is0 & (base["CPU"] <= CPU_IDLE_0GPU) & (base["Memory"] <= MEM_IDLE_0GPU)
idleg = (~is0) & (base["CPU"] <= CPU_IDLE_GPU) & (base["Memory"] <= MEM_IDLE_GPU) & (base["GPU"] <= GPU_IDLE_GPU)
idle_flag = (idle0 | idleg)

# filename 단위 has_used(any used)
has_used = (~idle_flag).groupby(base["filename"]).any()

# ✅ Normal filename만 유지 (idle row 포함)
normal_filenames = has_used[has_used].index
normal_data_sorted = (
    base[base["filename"].isin(normal_filenames)]
      .sort_values(["filename", "time"])
      .copy()
)


In [ ]:
_mf_capture_cell(30)


# Robust 강조


In [ ]:
_MF_CELL=32
df_input = normal_data_sorted


In [ ]:
_mf_capture_cell(32)


In [ ]:
_MF_CELL=33
# =========================================================
# 고정 3세트(Base + Sensitivity 2개)로 그림 그리기
#   - Base:        (0GPU: CPU10,MEM15) / (GPU: CPU10,MEM20,GPU=0)
#   - Sensitivity: (15,20) / (15,20,0)
#   - Sensitivity: (20,25) / (20,25,0)
# =========================================================

# ✅ 비교할 threshold 세트 정의
fixed_sets = [
    {
        "name": "BASE",
        "CPU0": 10, "MEM0": 15,
        "CPUG": 10, "MEMG": 20,
        "GPUG": 0
    },
    {
        "name": "S1",
        "CPU0": 10, "MEM0": 20,
        "CPUG": 10, "MEMG": 25,
        "GPUG": 0
    },
    {
        "name": "S2",
        "CPU0": 15, "MEM0": 20,
        "CPUG": 15, "MEMG": 20,
        "GPUG": 0
    },
    {
        "name": "S3",
        "CPU0": 20, "MEM0": 25,
        "CPUG": 20, "MEMG": 25,
        "GPUG": 0
    },
]


In [ ]:
_mf_capture_cell(33)


In [ ]:
_MF_CELL=34
def build_state_used_mask(df, CPU0, MEM0, CPUG, MEMG, GPUG=0):
    is_0gpu = (df["gpu_allocated(group)"] == "0 GPU")

    # idle 조건 (CPU/MEM은 임계 이하 + GPU는 0 고정)
    idle_0 = is_0gpu & (df["CPU"] <= CPU0) & (df["Memory"] <= MEM0)

    # GPU-Enabled: GPU<=0 엄격 + CPU/MEM 임계
    idle_g = (~is_0gpu) & (df["CPU"] <= CPUG) & (df["Memory"] <= MEMG) & (df["GPU"] <= GPUG)

    # used = not idle
    return ~(idle_0 | idle_g)

def build_pre_post(df_sorted, CPU0, MEM0, CPUG, MEMG, GPUG):
    state_used = build_state_used_mask(df_sorted, CPU0, MEM0, CPUG, MEMG, GPUG)

    used_rows = df_sorted.loc[state_used, ["filename", "used_time(minute)", "gpu_allocated(group)"]].copy()
    if len(used_rows) == 0:
        return None, None, None, None

    # runtime
    runtime = df_sorted.groupby("filename")["used_time(minute)"].max()

    # -------- Pre: first used time
    first_used_time = used_rows.groupby("filename")["used_time(minute)"].min()
    pre = first_used_time.to_frame("first_used_time").join(runtime.rename("runtime"), how="inner")
    pre["time_before_used"] = (pre["first_used_time"] - 1).clip(lower=0).astype(float)

    # gpu group per file
    gpu_group = df_sorted.groupby("filename")["gpu_allocated(group)"].last()
    pre = pre.join(gpu_group.rename("gpu_allocated(group)"), how="left").reset_index()

    # -------- Post: last used time
    last_used_time = used_rows.groupby("filename")["used_time(minute)"].max()
    post = last_used_time.to_frame("last_used_time").join(runtime.rename("runtime"), how="inner")
    post["idle_after_used"] = (post["runtime"] - post["last_used_time"]).clip(lower=0).astype(float)
    post = post.join(gpu_group.rename("gpu_allocated(group)"), how="left").reset_index()

    # =========================================================
    # ✅ 핵심 추가 조건:
    # "USED가 있었다가 idle-end로 떨어진 것만"
    #  - idle_after_used > 0 이어야 함
    #  - 그리고 마지막 row가 idle인지 확인하려면, 컨테이너의 마지막 row가 'idle'로 판정돼야 함
    # =========================================================
    # 마지막 row 인덱스 구하기
    tail_idx = df_sorted.groupby("filename").tail(1).set_index("filename").index
    tail_rows = df_sorted.groupby("filename").tail(1).set_index("filename")

    # tail row가 idle인지 여부 (state_used의 반대)
    # tail row에 대해서도 같은 idle 판정 로직을 적용
    tail_used_mask = build_state_used_mask(tail_rows.reset_index(), CPU0, MEM0, CPUG, MEMG, GPUG).to_numpy()
    # tail_used_mask == True면 tail이 used, False면 tail이 idle
    tail_is_idle = pd.Series(~tail_used_mask, index=tail_rows.index, name="tail_is_idle")

    # post에 붙이고 필터
    post = post.set_index("filename").join(tail_is_idle, how="left").reset_index()
    post = post[(post["idle_after_used"] > 0) & (post["tail_is_idle"] == True)].copy()

    return pre, post, runtime, used_rows

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import math

# =========================
# User params
# =========================
PCTS = [90, 95, 99]
IEEE_1COL_W = 3.5
IEEE_2COL_W = 7.16


def _gpu_bin(series_gpu_alloc):
    s = series_gpu_alloc.astype(str)
    return np.where(s.eq("0 GPU"), "Non-GPU", "GPU-Enabled")

def _safe_percentiles(values, pcts=PCTS, clip_min=1.0):
    v = pd.Series(values).dropna().astype(float).to_numpy()
    if v.size == 0:
        return {p: np.nan for p in pcts}
    v = np.clip(v, clip_min, None)
    return {p: float(np.percentile(v, p)) for p in pcts}

def _call_build_pre_post(df_sorted, CPU0, MEM0, CPUG, MEMG, GPUG):
    res = build_pre_post(df_sorted, CPU0, MEM0, CPUG, MEMG, GPUG)
    if not isinstance(res, (tuple, list)) or len(res) < 2:
        raise ValueError("build_pre_post() must return at least (pre_df, post_df).")
    return res[0], res[1]

def _short_setting_labels(fixed_sets, custom_map=None):
    names = [s["name"] for s in fixed_sets]
    out = []
    s_idx = 1
    for n in names:
        if custom_map and n in custom_map:
            out.append(custom_map[n])
        elif str(n).upper().startswith("BASE"):
            out.append("BASE")
        else:
            out.append(f"S{s_idx}")
            s_idx += 1
    return names, out

def _snap_ylim_decades(values, clip_min=1.0, pad_decades=0.12):
    v = pd.Series(values).dropna().astype(float)
    if v.empty:
        return (1, 10)
    v = v[v > 0].clip(lower=clip_min)
    if v.empty:
        return (1, 10)

    vmin, vmax = float(v.min()), float(v.max())
    log_min = math.log10(vmin) - pad_decades
    log_max = math.log10(vmax) + pad_decades
    low = 10 ** math.floor(log_min)
    high = 10 ** math.ceil(log_max)
    if high <= low:
        high = low * 10
    return (low, high)

def build_robustness_stats(normal_data_sorted, fixed_sets, pcts=PCTS):
    rows = []
    for s in fixed_sets:
        tag = s["name"]
        CPU0, MEM0, CPUG, MEMG, GPUG = s["CPU0"], s["MEM0"], s["CPUG"], s["MEMG"], s["GPUG"]

        pre_df, post_df = _call_build_pre_post(normal_data_sorted, CPU0, MEM0, CPUG, MEMG, GPUG)
        if pre_df is None or post_df is None or len(pre_df) == 0 or len(post_df) == 0:
            continue

        pre_df = pre_df.copy()
        post_df = post_df.copy()
        pre_df["gpu_bin"] = _gpu_bin(pre_df["gpu_allocated(group)"])
        post_df["gpu_bin"] = _gpu_bin(post_df["gpu_allocated(group)"])

        for dev in ["Non-GPU", "GPU-Enabled"]:
            pre_vals = pre_df.loc[pre_df["gpu_bin"] == dev, "time_before_used"]
            post_vals = post_df.loc[post_df["gpu_bin"] == dev, "idle_after_used"]

            pre_p = _safe_percentiles(pre_vals, pcts=pcts, clip_min=1.0)
            post_p = _safe_percentiles(post_vals, pcts=pcts, clip_min=1.0)

            for p in pcts:
                rows.append({"Setting": tag, "Device": dev, "Phase": "Pre",  "Percentile": p, "Value": pre_p[p]})
                rows.append({"Setting": tag, "Device": dev, "Phase": "Post", "Percentile": p, "Value": post_p[p]})

    return pd.DataFrame(rows)

def plot_robustness_4panel_ieee(
    df_stats,
    fixed_sets,
    pcts=PCTS,
    ieee_mode="2col",         # "1col" or "2col"
    fig_h=4.6,                # ✅ 여기만 키워도 체감 큼 (추천 4.2~5.2)
    font_pt=10,               # ✅ 글자 크기
    marker_size=9,            # ✅ 마커 크게
    line_w=2.0,               # ✅ 선 굵게
    custom_label_map=None,
    save=False,
    out_prefix="robustness_4panel_ieee"
):
    # --- labels
    setting_names, short_labels = _short_setting_labels(fixed_sets, custom_map=custom_label_map)

    df = df_stats.copy()
    df["Setting"] = pd.Categorical(df["Setting"], categories=setting_names, ordered=True)
    df = df.sort_values(["Phase", "Device", "Percentile", "Setting"])

    # --- figsize
    fig_w = IEEE_2COL_W if ieee_mode == "2col" else IEEE_1COL_W

    # ✅ rcParams를 함수 내부에서 확실히 고정 (이전 셀의 글로벌 폰트 영향 방지)
    with plt.rc_context({
        "font.size": font_pt,
        "axes.titlesize": font_pt,
        "axes.labelsize": font_pt,
        "legend.fontsize": font_pt - 1,
        "xtick.labelsize": font_pt,
        "ytick.labelsize": font_pt,
    }):
        # ✅ constrained_layout이 2x2 패널에서 안정적
        fig, axes = plt.subplots(2, 2, figsize=(fig_w, fig_h), constrained_layout=True, sharex=True)

        ax_map = {
            ("Pre",  "Non-GPU"):      axes[0, 0],
            ("Pre",  "GPU-Enabled"):  axes[0, 1],
            ("Post", "Non-GPU"):      axes[1, 0],
            ("Post", "GPU-Enabled"):  axes[1, 1],
        }

        marker_map = {90: "o", 95: "s", 99: "^"}
        ls_map     = {90: "-", 95: "--", 99: ":"}

        # --- y-lim 고정 (phase별)
        pre_ylim  = _snap_ylim_decades(df.loc[df["Phase"]=="Pre",  "Value"], clip_min=1.0)
        post_ylim = _snap_ylim_decades(df.loc[df["Phase"]=="Post", "Value"], clip_min=1.0)

        x = np.arange(len(setting_names))

        for phase in ["Pre", "Post"]:
            for dev in ["Non-GPU", "GPU-Enabled"]:
                ax = ax_map[(phase, dev)]
                for p in pcts:
                    d = df[(df["Phase"] == phase) & (df["Device"] == dev) & (df["Percentile"] == p)].copy()
                    d = d.set_index("Setting").reindex(setting_names).reset_index()
                    y = d["Value"].astype(float).to_numpy()

                    ax.plot(
                        x, y,
                        color="black",
                        linestyle=ls_map[p],
                        marker=marker_map[p],
                        linewidth=line_w,
                        markersize=marker_size,
                        label=f"P{p}"
                    )

                ax.set_yscale("log")
                ax.grid(True, which="both", axis="y", linestyle="--", alpha=0.25)
                ax.set_title(f"{phase}: {dev}")

                ax.set_ylim(pre_ylim if phase == "Pre" else post_ylim)

                if dev == "Non-GPU":
                    ax.set_ylabel("Idle time (min, log)")

        # x-axis
        for ax in axes[1, :]:
            ax.set_xticks(x)
            ax.set_xticklabels(short_labels, rotation=0)
            ax.set_xlabel("Threshold setting")

        # legend: 좌상단 한 곳만
        axes[0, 0].legend(
    loc="upper left",
    ncol=len(pcts),          # ✅ 가로로 나열 (P90 P95 P99)
    frameon=True,
    title=None,              # ✅ "Percentile" 제거
    handlelength=2.0,
    columnspacing=1.0,
    borderaxespad=0.3
)


        if save:
            fig.savefig(f"{out_prefix}.png", dpi=300, bbox_inches="tight")
            fig.savefig(f"{out_prefix}.pdf", bbox_inches="tight")

        plt.show()


# =========================
# Run
# =========================
df_stats = build_robustness_stats(normal_data_sorted, fixed_sets, pcts=PCTS)

# ✅ 크기 추천값: fig_h=4.6 (2col 기준)
plot_robustness_4panel_ieee(
    df_stats,
    fixed_sets,
    pcts=PCTS,
    ieee_mode="2col",
    fig_h=4.8,          # ← 더 키우고 싶으면 5.2까지
    font_pt=11,         # ← 글자가 너무 크면 10
    marker_size=7,     # ← 마커 더 크면 11~12
    line_w=2.2,
    save=False
)


In [ ]:
_mf_capture_cell(34)


# FIGURE 10
# FIGURE 11
## 0405_normal_container_idle_time_distribution
## 0408_normal_container_idle_time_cdf


# Pre, POST hoarding time 계산


In [ ]:
_MF_CELL=37
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

try:
    from IPython.display import display
except Exception:
    display = print


# =========================================================
# Core: schema & masks
# =========================================================
def _must_have_cols(df, cols):
    miss = [c for c in cols if c not in df.columns]
    if miss:
        raise KeyError(f"필수 컬럼이 없습니다: {miss}")

def prepare_timeseries_df(df):
    """최소한의 안전한 정리: 타입/정렬."""
    need = ["filename", "used_time(minute)", "gpu_allocated(group)", "CPU", "Memory", "GPU"]
    _must_have_cols(df, need)

    x = df.copy()
    x["filename"] = x["filename"].astype(str).str.strip()
    x["used_time(minute)"] = pd.to_numeric(x["used_time(minute)"], errors="coerce")
    x["CPU"] = pd.to_numeric(x["CPU"], errors="coerce")
    x["Memory"] = pd.to_numeric(x["Memory"], errors="coerce")
    x["GPU"] = pd.to_numeric(x["GPU"], errors="coerce")

    x = x.dropna(subset=["filename", "used_time(minute)"]).copy()
    x = x.sort_values(["filename", "used_time(minute)"]).reset_index(drop=True)
    return x

def gpu_bin_from_group(series_gpu_alloc):
    s = series_gpu_alloc.astype(str)
    return np.where(s.eq("0 GPU"), "Non-GPU", "GPU-Enabled")

def build_state_used_mask(df, CPU0, MEM0, CPUG, MEMG, GPUG=0.0):
    """
    used = not idle
      - Non-GPU(0 GPU): CPU<=CPU0 & MEM<=MEM0 => idle
      - GPU-Enabled(>0 GPU): CPU<=CPUG & MEM<=MEMG & GPU<=GPUG => idle
    """
    is_0gpu = df["gpu_allocated(group)"].astype(str).eq("0 GPU")
    cpu = pd.to_numeric(df["CPU"], errors="coerce")
    mem = pd.to_numeric(df["Memory"], errors="coerce")
    gpu = pd.to_numeric(df["GPU"], errors="coerce")

    idle_0 = is_0gpu & (cpu <= CPU0) & (mem <= MEM0)
    idle_g = (~is_0gpu) & (cpu <= CPUG) & (mem <= MEMG) & (gpu <= GPUG)
    return ~(idle_0 | idle_g)


# =========================================================
# Pre/Post(EOL) (container-level)
# =========================================================
def build_prepost_eol(
    df_sorted,
    CPU0, MEM0, CPUG, MEMG, GPUG=0.0,
    min_pre_idle_min=1.0,
    min_post_idle_min=1.0,
):
    """
    normal containers(=has_any_used) 기준
      - Pre: start~first USED
      - Post(EOL): last USED~end, tail_is_idle True
    """
    x = prepare_timeseries_df(df_sorted)

    x["is_used"] = build_state_used_mask(x, CPU0, MEM0, CPUG, MEMG, GPUG)

    # per-container gpu_bin (last)
    last_gpu_group = x.groupby("filename")["gpu_allocated(group)"].last()
    gpu_bin = pd.Series(gpu_bin_from_group(last_gpu_group), index=last_gpu_group.index, name="gpu_bin")

    g = x.groupby("filename", sort=False)
    start_t = g["used_time(minute)"].min()
    end_t   = g["used_time(minute)"].max()

    first_used = x.loc[x["is_used"]].groupby("filename")["used_time(minute)"].min()
    last_used  = x.loc[x["is_used"]].groupby("filename")["used_time(minute)"].max()

    # normal containers = has_any_used
    normal_files = first_used.index.intersection(end_t.index)

    out = pd.DataFrame({
        "filename": normal_files,
        "start_min": start_t.loc[normal_files].values,
        "end_min": end_t.loc[normal_files].values,
        "first_used_min": first_used.loc[normal_files].values,
        "last_used_min": last_used.loc[normal_files].values,
    }).set_index("filename")

    out["gpu_bin"] = gpu_bin.loc[normal_files].values

    out["pre_idle_min"] = (out["first_used_min"] - out["start_min"]).clip(lower=0).astype(float)
    out["post_eol_idle_min"] = (out["end_min"] - out["last_used_min"]).clip(lower=0).astype(float)

    # tail_is_idle (마지막 row가 idle인지)
    tail_rows = x.groupby("filename").tail(1).set_index("filename").loc[normal_files]
    tail_used_mask = build_state_used_mask(tail_rows.reset_index(), CPU0, MEM0, CPUG, MEMG, GPUG).to_numpy()
    out["tail_is_idle"] = (~tail_used_mask).astype(bool)

    # valid cases (>=1min + post는 tail_is_idle)
    out["is_pre_case"] = out["pre_idle_min"].notna() & (out["pre_idle_min"] >= min_pre_idle_min)
    out["is_post_case"] = (
        out["post_eol_idle_min"].notna()
        & (out["post_eol_idle_min"] >= min_post_idle_min)
        & (out["tail_is_idle"] == True)
    )

    return out.reset_index()


# =========================================================
# Tables
# =========================================================
def percentile_list(arr, pcts):
    arr = pd.Series(arr).dropna().to_numpy(dtype=float)
    if len(arr) == 0:
        return [np.nan] * len(pcts)
    return [float(np.percentile(arr, p)) for p in pcts]

def summarize_table(df_prepost, pcts, tail_mins):
    """
    - normal_containers: has_any_used 기준 분모
    - Pre/Post Pxx: valid 케이스(>=1min)에서만 계산
    - Tail ratio: 분모=normal_containers, 0 포함 전체에서 계산
    """
    rows = []
    for bin_name in ["Non-GPU", "GPU-Enabled", "Total"]:
        d = df_prepost if bin_name == "Total" else df_prepost[df_prepost["gpu_bin"] == bin_name]
        normal_n = d["filename"].nunique()

        pre_vals  = d.loc[d["is_pre_case"], "pre_idle_min"].to_numpy(dtype=float)
        post_vals = d.loc[d["is_post_case"], "post_eol_idle_min"].to_numpy(dtype=float)

        r = {"gpu_bin": bin_name, "normal_containers": float(normal_n)}

        pre_pct  = percentile_list(pre_vals, pcts)
        post_pct = percentile_list(post_vals, pcts)

        for p, v in zip(pcts, pre_pct):
            r[f"Pre P{p} (min)"] = v
        for p, v in zip(pcts, post_pct):
            r[f"Post P{p} (min)"] = v

        pre_all  = d["pre_idle_min"].fillna(0).to_numpy(dtype=float)
        post_all = d["post_eol_idle_min"].fillna(0).to_numpy(dtype=float)

        for thr in tail_mins:
            r[f"Pre >= {int(thr)}min_normal(%)"]  = (np.mean(pre_all  >= thr) * 100.0) if normal_n > 0 else np.nan
            r[f"Post >= {int(thr)}min_normal(%)"] = (np.mean(post_all >= thr) * 100.0) if normal_n > 0 else np.nan

        rows.append(r)

    return pd.DataFrame(rows).set_index("gpu_bin").round(2)

def split_pre_post_tables(tbl_all, pcts, tail_mins):
    idx_order = ["Non-GPU", "GPU-Enabled", "Total"]
    tbl_all = tbl_all.reindex([i for i in idx_order if i in tbl_all.index])

    pre_cols = (
        ["normal_containers"]
        + [f"Pre P{p} (min)" for p in pcts]
        + [f"Pre >= {int(thr)}min_normal(%)" for thr in tail_mins]
    )
    post_cols = (
        ["normal_containers"]
        + [f"Post P{p} (min)" for p in pcts]
        + [f"Post >= {int(thr)}min_normal(%)" for thr in tail_mins]
    )

    pre_tbl  = tbl_all.loc[:, pre_cols].copy().round(2)
    post_tbl = tbl_all.loc[:, post_cols].copy().round(2)
    return pre_tbl, post_tbl


# =========================================================
# Plot helpers
# =========================================================
def plot_violin_2panel(pre_long, post_long, y_lim=(1e-1, 1e6)):
    order2 = ["Non-GPU", "GPU-Enabled"]
    fig, axes = plt.subplots(1, 2, figsize=(30, 8))

    sns.violinplot(
        data=pre_long, x="gpu_bin", y="value",
        order=order2, ax=axes[0],
        log_scale=(False, True), inner="box", cut=2, bw_adjust=0.6, scale="width"
    )
    axes[0].set_title("Pre-Execution Idle Time", fontsize=35)
    axes[0].set_xlabel("Device Type", fontsize=35)
    axes[0].set_ylabel("Time (minutes)", fontsize=35)
    axes[0].tick_params(axis="x", labelsize=35)
    axes[0].tick_params(axis="y", labelsize=35)
    axes[0].grid(False)

    sns.violinplot(
        data=post_long, x="gpu_bin", y="value",
        order=order2, ax=axes[1],
        log_scale=(False, True), inner="box", cut=2, bw_adjust=0.6, scale="width"
    )
    axes[1].set_title("Post-Execution Idle Time", fontsize=35)
    axes[1].set_xlabel("Device Type", fontsize=35)
    axes[1].set_ylabel("Time (minutes)", fontsize=35)
    axes[1].tick_params(axis="x", labelsize=35)
    axes[1].tick_params(axis="y", labelsize=35)
    axes[1].grid(False)

    for ax in axes:
        ax.yaxis.set_major_locator(mtick.LogLocator(base=10))
        ax.yaxis.set_minor_locator(mtick.LogLocator(base=10, subs=range(2, 10)))
        ax.yaxis.set_major_formatter(mtick.LogFormatterMathtext(base=10))
        ax.set_ylim(*y_lim)

    plt.tight_layout()
    plt.show()

def pct_tbl_from_long(long_df, pcts, bins=("Non-GPU", "GPU-Enabled")):
    def pct(arr):
        arr = pd.Series(arr).dropna().to_numpy(dtype=float)
        if len(arr) == 0:
            return [np.nan] * len(pcts)
        return [float(np.percentile(arr, p)) for p in pcts]

    out = pd.DataFrame({"Percentile (%)": pcts})
    for b in bins:
        out[b] = pct(long_df.loc[long_df["gpu_bin"] == b, "value"])
    return out.round(2)

def plot_percentile_2panel(pre_tbl2, post_tbl2):
    fig, axes = plt.subplots(1, 2, figsize=(30, 8))

    for ax, tbl, ttl in [
        (axes[0], pre_tbl2,  "Pre-Execution Idle Time"),
        (axes[1], post_tbl2, "Post-Execution Idle Time"),
    ]:
        tmp = tbl.copy()
        for col in ["Non-GPU", "GPU-Enabled"]:
            tmp[col] = pd.to_numeric(tmp[col], errors="coerce").clip(lower=1.0)

        for col in ["Non-GPU", "GPU-Enabled"]:
            ax.plot(tmp["Percentile (%)"], tmp[col], marker="o", markersize=14, linewidth=5, label=col)

        ax.set_title(ttl, fontsize=35)
        ax.set_xlabel("Percentile (%)", fontsize=35)
        ax.set_ylabel("Time (minutes)", fontsize=35)
        ax.tick_params(axis="x", labelsize=35)
        ax.tick_params(axis="y", labelsize=35)
        ax.legend(fontsize=30, loc="upper left")

        ax.set_xticks(tmp["Percentile (%)"].tolist())
        ax.set_xlim(min(tmp["Percentile (%)"]) - 1, 99.5)

        ax.set_yscale("log")
        ax.yaxis.set_major_locator(mtick.LogLocator(base=10))
        ax.yaxis.set_minor_locator(mtick.LogLocator(base=10, subs=range(2, 10)))
        ax.yaxis.set_major_formatter(mtick.LogFormatterMathtext(base=10))
        ax.grid(False)

    plt.tight_layout()
    plt.show()

def _ecdf(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) == 0:
        return np.array([]), np.array([])
    arr = np.sort(arr)
    y = np.arange(1, len(arr) + 1) / len(arr)
    return arr, y

def plot_cdf_2panel(pre_vals_by_bin, post_vals_by_bin, bins=("Non-GPU", "GPU-Enabled")):
    fig, axes = plt.subplots(1, 2, figsize=(30, 8))

    ax = axes[0]
    for b in bins:
        x, y = _ecdf(pre_vals_by_bin.get(b, []))
        if len(x) == 0:
            continue
        ax.step(x, y, where="post", linewidth=5, label=b)
    ax.set_title("Pre-Execution Idle Time (CDF)", fontsize=35)
    ax.set_xlabel("Time (minutes)", fontsize=35)
    ax.set_ylabel("CDF", fontsize=35)
    ax.tick_params(axis="x", labelsize=35)
    ax.tick_params(axis="y", labelsize=35)
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(mtick.LogLocator(base=10))
    ax.xaxis.set_major_formatter(mtick.LogFormatterMathtext(base=10))
    ax.set_xlim(1, 1e6)
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=30, loc="lower right")
    ax.grid(False)

    ax = axes[1]
    for b in bins:
        x, y = _ecdf(post_vals_by_bin.get(b, []))
        if len(x) == 0:
            continue
        ax.step(x, y, where="post", linewidth=5, label=b)
    ax.set_title("Post-Execution Idle Time (EOL) (CDF)", fontsize=35)
    ax.set_xlabel("Time (minutes)", fontsize=35)
    ax.set_ylabel("CDF", fontsize=35)
    ax.tick_params(axis="x", labelsize=35)
    ax.tick_params(axis="y", labelsize=35)
    ax.set_xscale("log")
    ax.xaxis.set_major_locator(mtick.LogLocator(base=10))
    ax.xaxis.set_major_formatter(mtick.LogFormatterMathtext(base=10))
    ax.set_xlim(1, 1e6)
    ax.set_ylim(0, 1.02)
    ax.legend(fontsize=30, loc="lower right")
    ax.grid(False)

    plt.tight_layout()
    plt.show()


# =========================================================
# Top-share wide tables
# =========================================================
def top_share_wide_table(df, value_cols, top_pcts=(1, 5, 10, 100),
                         id_col="filename", group_name="Total", col_alias=None):
    x = df.copy()
    x = x[x[id_col].notna()].copy()

    for c in value_cols:
        x[c] = pd.to_numeric(x[c], errors="coerce").fillna(0.0)

    # container-level 1행이 이상적. 중복이면 합으로 축약
    if x.duplicated(subset=[id_col]).any():
        agg = {c: "sum" for c in value_cols}
        x = x.groupby(id_col, as_index=False).agg(agg)

    sort_key = x[value_cols].sum(axis=1) if len(value_cols) > 1 else x[value_cols[0]]
    x = x.assign(_sort_key=sort_key).sort_values("_sort_key", ascending=False).drop(columns="_sort_key")

    n_total = x[id_col].nunique()
    totals = {c: float(x[c].sum()) for c in value_cols}

    rows = []
    for p in top_pcts:
        k = int(np.ceil(n_total * (p / 100.0))) if n_total > 0 else 0
        top = x.head(k) if k > 0 else x.iloc[:0]

        r = {"Top": f"{int(p)}%", "n_containers": k, "group": group_name}
        for c in value_cols:
            prefix = col_alias.get(c, c) if isinstance(col_alias, dict) else c
            s = float(top[c].sum()) if k > 0 else 0.0
            share = (s / totals[c] * 100.0) if totals[c] > 0 else np.nan
            r[f"{prefix}_sum"] = round(s, 2)
            r[f"{prefix}_share_%"] = round(share, 2) if not np.isnan(share) else np.nan

        rows.append(r)

    return pd.DataFrame(rows).set_index("Top")

def top_share_wide_by_bin(df, value_cols, top_pcts=(1, 5, 10, 100),
                          group_col="gpu_bin", include_total=True, col_alias=None):
    tables = {}
    if group_col in df.columns:
        for gname, d in df.groupby(group_col, sort=False):
            tables[gname] = top_share_wide_table(
                d, value_cols=value_cols, top_pcts=top_pcts, group_name=gname, col_alias=col_alias
            )
    if include_total:
        tables["Total"] = top_share_wide_table(
            df, value_cols=value_cols, top_pcts=top_pcts, group_name="Total", col_alias=col_alias
        )
    return tables


# =========================================================
# Single setting runner (외부 전역 0개)
# =========================================================
def run_one_setting(
    df,
    setting,
    *,
    pcts=None,
    tail_hours=(3, 5, 10, 12, 24, 72),
    min_pre_idle_min=1.0,
    min_post_idle_min=1.0,
    plot_bins=("Non-GPU", "GPU-Enabled"),
    do_plots=True,
    do_topshare=True,
):
    """
    반환:
      {
        "df_prepost": ...,
        "pre_tbl": ...,
        "post_tbl": ...,
        "pre_pct_tbl": ...,
        "post_pct_tbl": ...,
        "pre_topshare": {...},
        "post_topshare": {...},
        "checks": {...}
      }
    """
    if pcts is None:
        pcts = list(range(60, 100, 5)) + [99]
    tail_mins = [int(h * 60) for h in tail_hours]

    CPU0, MEM0, CPUG, MEMG, GPUG = (
        setting["CPU0"], setting["MEM0"], setting["CPUG"], setting["MEMG"], setting.get("GPUG", 0.0)
    )
    tag = setting.get("name", "SETTING")

    df_prepost = build_prepost_eol(
        df, CPU0, MEM0, CPUG, MEMG, GPUG,
        min_pre_idle_min=min_pre_idle_min,
        min_post_idle_min=min_post_idle_min
    )

    df_sorted = prepare_timeseries_df(df)
    containers_total = df_sorted["filename"].nunique()
    normal_containers = df_prepost["filename"].nunique()
    never_used = containers_total - normal_containers

    tbl_all = summarize_table(df_prepost, pcts, tail_mins)
    pre_tbl, post_tbl = split_pre_post_tables(tbl_all, pcts, tail_mins)

    # plot용 long (>=1min, bins만)
    pre_plot = df_prepost[df_prepost["is_pre_case"] & df_prepost["gpu_bin"].isin(plot_bins)].copy()
    post_plot = df_prepost[df_prepost["is_post_case"] & df_prepost["gpu_bin"].isin(plot_bins)].copy()

    pre_long = pre_plot[["gpu_bin", "pre_idle_min"]].rename(columns={"pre_idle_min": "value"})
    post_long = post_plot[["gpu_bin", "post_eol_idle_min"]].rename(columns={"post_eol_idle_min": "value"})
    pre_long["value"] = pd.to_numeric(pre_long["value"], errors="coerce").clip(lower=1.0)
    post_long["value"] = pd.to_numeric(post_long["value"], errors="coerce").clip(lower=1.0)

    pre_pct_tbl = pct_tbl_from_long(pre_long, pcts, bins=plot_bins)
    post_pct_tbl = pct_tbl_from_long(post_long, pcts, bins=plot_bins)

    if do_plots:
        print(f"\n[{tag}] 0GPU: CPU<={CPU0}, MEM<={MEM0} | GPU: CPU<={CPUG}, MEM<={MEMG}, GPU<={GPUG}")
        print(f"[CHECK] containers_total(all filenames) = {containers_total}")
        print(f"[CHECK] normal_containers(has_any_used) = {normal_containers}")
        print(f"[CHECK] never-used(containers with NO USED) = {never_used}")

        print("\n=== PRE table ===")
        display(pre_tbl)
        print("\n=== POST table ===")
        display(post_tbl)

        plot_violin_2panel(pre_long, post_long)
        plot_percentile_2panel(pre_pct_tbl, post_pct_tbl)

        pre_vals_by_bin  = {b: pre_long.loc[pre_long["gpu_bin"] == b, "value"].to_numpy(dtype=float) for b in plot_bins}
        post_vals_by_bin = {b: post_long.loc[post_long["gpu_bin"] == b, "value"].to_numpy(dtype=float) for b in plot_bins}
        plot_cdf_2panel(pre_vals_by_bin, post_vals_by_bin, bins=plot_bins)

        display(pre_pct_tbl)
        display(post_pct_tbl)

    pre_topshare = {}
    post_topshare = {}
    if do_topshare:
        TOP_PCTS = (1, 5, 10, 100)
        pre_df = df_prepost[df_prepost["is_pre_case"]].copy()
        post_df = df_prepost[df_prepost["is_post_case"]].copy()

        pre_topshare = top_share_wide_by_bin(
            pre_df, value_cols=["pre_idle_min"], top_pcts=TOP_PCTS,
            col_alias={"pre_idle_min": "PreIdleMin"}
        )
        post_topshare = top_share_wide_by_bin(
            post_df, value_cols=["post_eol_idle_min"], top_pcts=TOP_PCTS,
            col_alias={"post_eol_idle_min": "PostEOLIdleMin"}
        )

        if do_plots:
            print("\n[Top-share Wide] Pre (>=1min)")
            for k, t in pre_topshare.items():
                print(f"\n--- {k} ---")
                display(t)

            print("\n[Top-share Wide] Post(EOL) (tail_is_idle)")
            for k, t in post_topshare.items():
                print(f"\n--- {k} ---")
                display(t)

    return {
        "df_prepost": df_prepost,
        "pre_tbl": pre_tbl,
        "post_tbl": post_tbl,
        "pre_pct_tbl": pre_pct_tbl,
        "post_pct_tbl": post_pct_tbl,
        "pre_topshare": pre_topshare,
        "post_topshare": post_topshare,
        "checks": {
            "containers_total": containers_total,
            "normal_containers": normal_containers,
            "never_used": never_used
        }
    }


def run_analysis(
    df,
    fixed_sets,
    *,
    setting_used_name="BASE",
    run_all_settings=False,
    **kwargs
):
    """
    df만 넘기면 끝. 외부 전역 참조 없음.
    반환: {setting_name: result_dict}
    """
    sets = fixed_sets if run_all_settings else [next(s for s in fixed_sets if s.get("name") == setting_used_name)]
    out = {}
    for s in sets:
        out[s.get("name", "SETTING")] = run_one_setting(df, s, **kwargs)
    return out


# =========================================================
# 사용 예시 (정확히 여기서만 normal_data_sorted를 씁니다)
# =========================================================
results = run_analysis(
    normal_data_sorted,         # <- 데이터셋(전역이어도 OK, 함수는 전역 참조 안 함)
    fixed_sets,
    setting_used_name="BASE",
    run_all_settings=False,
    min_pre_idle_min=1.0,
    min_post_idle_min=1.0,
    pcts=list(range(60, 100, 5)) + [99],
    tail_hours=(3, 5, 10, 12, 24, 72),
    do_plots=True,
    do_topshare=True
)


In [ ]:

# Freeze the original BASE result before subsequent exploratory cells overwrite it.
_paper_base=results['BASE']
_paper_tables=ROOT/'05_Results/section4/tables'; _paper_tables.mkdir(parents=True,exist_ok=True)
for _name in ['pre_tbl','post_tbl','pre_pct_tbl','post_pct_tbl']:
    _paper_base[_name].to_csv(_paper_tables/('Figure10_11_'+_name+'.csv'),index=True)
_paper_base['df_prepost'].to_parquet(_paper_tables/'Figure10_11_prepost_cases.parquet',index=False)
_share_rows=[]
for _phase,_field in [('Pre','pre_topshare'),('Post','post_topshare')]:
    for _group,_table in _paper_base[_field].items():
        for _top,_row in _table.iterrows():
            _share_col=next(c for c in _table.columns if c.endswith('share_%'))
            _sum_col=next(c for c in _table.columns if c.endswith('_sum'))
            _share_rows.append({'phase':_phase,'group':_group,'top_percent':float(str(_top).rstrip('%')),
                                'n_containers':int(_row['n_containers']),
                                'idle_minutes':float(_row[_sum_col]),'share_percent':float(_row[_share_col])})
_mf_pd.DataFrame(_share_rows).to_csv(_paper_tables/'Table08_idle_concentration.csv',index=False)
_paper_q=[]
for _phase,_value,_flag,_p in [('Pre','pre_idle_min','is_pre_case',90),('Post','post_eol_idle_min','is_post_case',70)]:
    for _group,_g in _paper_base['df_prepost'].groupby('gpu_bin'):
        _v=_g.loc[_g[_flag],_value]
        _paper_q.append({'phase':_phase,'group':_group,'percentile':_p,'cases':len(_v),
                         'raw_minutes':float(_v.quantile(_p/100)),
                         'rounded_minutes':int(round(_v.quantile(_p/100)))})
_mf_pd.DataFrame(_paper_q).to_csv(_paper_tables/'Figure11_threshold_recalculation.csv',index=False)
(_paper_tables/'Figure10_11_definition.json').write_text(_mf_json.dumps({
    'source_original_cell_index':37,'setting':'BASE','checks':_paper_base['checks'],
    'input':'03_ProcessedData/TimeSeriesData_train.parquet',
    'pre_time_origin':'first retained elapsed sample','post_time_origin':'last active sample',
    'minimum_pre_minutes':1,'minimum_post_minutes':1,
    'policy_timeouts_frozen':{'pre_non_gpu':85,'pre_integer_gpu':119,'post_non_gpu':146,'post_integer_gpu':170},
    'recalculated_percentiles_do_not_change_policy':True},ensure_ascii=False,indent=2))


In [ ]:
_mf_capture_cell(37)


In [ ]:
_MF_CELL=38



In [ ]:
_mf_capture_cell(38)


In [ ]:
_MF_CELL=39



In [ ]:
_mf_capture_cell(39)


In [ ]:
_MF_CELL=40
# log스케일 사용
# gpu_allocated(group) 나누지 않고 Non-GPU, GPU-Enabled만

# =========================================================
# [A] Normal 데이터(정렬)는 이미 만들어둔 걸 그대로 사용
#  - normal_data_sorted: df_3에서 is_lowutil=='normal'만 필터 후 filename,time 정렬된 DF
# =========================================================

# =========================================================
# [B] helper: state_used mask 만들기 (벡터화)
#  - GPU idle threshold는 "0" 고정(엄격) 조건을 반영
# =========================================================
def build_state_used_mask(df, CPU0, MEM0, CPUG, MEMG, GPUG=0):
    is_0gpu = (df["gpu_allocated(group)"] == "0 GPU")

    # idle 조건 (CPU/MEM은 임계 이하 + GPU는 0 고정)
    idle_0 = is_0gpu & (df["CPU"] <= CPU0) & (df["Memory"] <= MEM0)

    # GPU-Enabled: GPU<=0 엄격 + CPU/MEM 임계
    idle_g = (~is_0gpu) & (df["CPU"] <= CPUG) & (df["Memory"] <= MEMG) & (df["GPU"] <= GPUG)

    # used = not idle
    return ~(idle_0 | idle_g)

# =========================================================
# [C] helper: Pre(첫 used 전 idle) & Post(마지막 used 이후 idle) DF 생성
#  - Post는 "USED가 있었다가 idle-end로 떨어진 컨테이너만" 포함하도록 추가 조건
# =========================================================
def build_pre_post(df_sorted, CPU0, MEM0, CPUG, MEMG, GPUG):
    state_used = build_state_used_mask(df_sorted, CPU0, MEM0, CPUG, MEMG, GPUG)

    used_rows = df_sorted.loc[state_used, ["filename", "used_time(minute)", "gpu_allocated(group)"]].copy()
    if len(used_rows) == 0:
        return None, None, None, None

    # runtime
    runtime = df_sorted.groupby("filename")["used_time(minute)"].max()

    # -------- Pre: first used time
    first_used_time = used_rows.groupby("filename")["used_time(minute)"].min()
    pre = first_used_time.to_frame("first_used_time").join(runtime.rename("runtime"), how="inner")
    pre["time_before_used"] = (pre["first_used_time"] - 1).clip(lower=0).astype(float)

    # gpu group per file
    gpu_group = df_sorted.groupby("filename")["gpu_allocated(group)"].last()
    pre = pre.join(gpu_group.rename("gpu_allocated(group)"), how="left").reset_index()

    # -------- Post: last used time
    last_used_time = used_rows.groupby("filename")["used_time(minute)"].max()
    post = last_used_time.to_frame("last_used_time").join(runtime.rename("runtime"), how="inner")
    post["idle_after_used"] = (post["runtime"] - post["last_used_time"]).clip(lower=0).astype(float)
    post = post.join(gpu_group.rename("gpu_allocated(group)"), how="left").reset_index()

    # =========================================================
    # ✅ 핵심 추가 조건:
    # "USED가 있었다가 idle-end로 떨어진 것만"
    #  - idle_after_used > 0 이어야 함
    #  - 그리고 마지막 row가 idle인지 확인하려면, 컨테이너의 마지막 row가 'idle'로 판정돼야 함
    # =========================================================
    # 마지막 row 인덱스 구하기
    tail_idx = df_sorted.groupby("filename").tail(1).set_index("filename").index
    tail_rows = df_sorted.groupby("filename").tail(1).set_index("filename")

    # tail row가 idle인지 여부 (state_used의 반대)
    # tail row에 대해서도 같은 idle 판정 로직을 적용
    tail_used_mask = build_state_used_mask(tail_rows.reset_index(), CPU0, MEM0, CPUG, MEMG, GPUG).to_numpy()
    # tail_used_mask == True면 tail이 used, False면 tail이 idle
    tail_is_idle = pd.Series(~tail_used_mask, index=tail_rows.index, name="tail_is_idle")

    # post에 붙이고 필터
    post = post.set_index("filename").join(tail_is_idle, how="left").reset_index()
    post = post[(post["idle_after_used"] > 0) & (post["tail_is_idle"] == True)].copy()

    return pre, post, runtime, used_rows

# =========================================================
# [D] helper: 2그룹(Non-GPU vs GPU-Enabled) 바이올린 + percentile curve
# =========================================================
def plot_2panel(pre_df, post_df, title_prefix=""):
    # 2그룹 bin
    pre2 = pre_df[["filename","gpu_allocated(group)","time_before_used"]].copy()
    pre2["gpu_bin"] = np.where(pre2["gpu_allocated(group)"]=="0 GPU", "Non-GPU", "GPU-Enabled")
    pre2["time_before_used"] = pre2["time_before_used"].astype(float).clip(lower=1.0)

    post2 = post_df[["filename","gpu_allocated(group)","idle_after_used"]].copy()
    post2["gpu_bin"] = np.where(post2["gpu_allocated(group)"]=="0 GPU", "Non-GPU", "GPU-Enabled")
    post2["idle_after_used"] = post2["idle_after_used"].astype(float).clip(lower=1.0)

    order2 = ["Non-GPU","GPU-Enabled"]

    # ---------- (1) Violin 2패널 ----------
    fig, axes = plt.subplots(1, 2, figsize=(30, 8))

    sns.violinplot(
        data=pre2, x="gpu_bin", y="time_before_used",
        order=order2, ax=axes[0],
        log_scale=(False, True), inner="box", cut=2, bw_adjust=0.6, scale="width"
    )
    # axes[0].set_title(f"{title_prefix} Pre-Execution Idle", fontsize=35)
    axes[0].set_title("Pre-Execution Idle Time", fontsize=35)
    axes[0].set_xlabel("Device Type", fontsize=35)
    axes[0].set_ylabel("Time (minutes)", fontsize=35)
    axes[0].tick_params(axis="x", labelsize=35)
    axes[0].tick_params(axis="y", labelsize=35)

    sns.violinplot(
        data=post2, x="gpu_bin", y="idle_after_used",
        order=order2, ax=axes[1],
        log_scale=(False, True), inner="box", cut=2, bw_adjust=0.6, scale="width"
    )
    # axes[1].set_title(f"{title_prefix} Post-Execution Idle", fontsize=35)
    axes[1].set_title("Post-Execution Idle Time", fontsize=35)
    axes[1].set_xlabel("Device Type", fontsize=35)
    axes[1].set_ylabel("Time (minutes)", fontsize=35)
    axes[1].tick_params(axis="x", labelsize=35)
    axes[1].tick_params(axis="y", labelsize=35)

    for ax in axes:
        ax.yaxis.set_major_locator(mtick.LogLocator(base=10))
        ax.yaxis.set_minor_locator(mtick.LogLocator(base=10, subs=range(2,10)))
        ax.yaxis.set_major_formatter(mtick.LogFormatterMathtext(base=10))
        ax.set_ylim(1e-1, 1e6)

    plt.tight_layout()
    plt.show()

    # ---------- (2) Percentile curve 2패널 ----------
    percentiles = [50, 60, 70, 80, 90, 95]
    def pct(arr):
        arr = pd.Series(arr).dropna().to_numpy(dtype=float)
        if len(arr) == 0:
            return [np.nan]*len(percentiles)
        return [float(np.percentile(arr, p)) for p in percentiles]

    pre_tbl2 = pd.DataFrame({
        "Percentile (%)": percentiles,
        "Non-GPU":     pct(pre2.loc[pre2["gpu_bin"]=="Non-GPU", "time_before_used"]),
        "GPU-Enabled": pct(pre2.loc[pre2["gpu_bin"]=="GPU-Enabled", "time_before_used"]),
    })
    post_tbl2 = pd.DataFrame({
        "Percentile (%)": percentiles,
        "Non-GPU":     pct(post2.loc[post2["gpu_bin"]=="Non-GPU", "idle_after_used"]),
        "GPU-Enabled": pct(post2.loc[post2["gpu_bin"]=="GPU-Enabled", "idle_after_used"]),
    })

    fig, axes = plt.subplots(1, 2, figsize=(30, 8))

    ax1 = axes[0]
    for col in ["Non-GPU","GPU-Enabled"]:
        ax1.plot(pre_tbl2["Percentile (%)"], pre_tbl2[col], marker="o", markersize=14, linewidth=5, label=col)
    # ax1.set_title(f"{title_prefix} Pre-Execution Idle (Percentiles)", fontsize=35)
    ax1.set_title("Pre-Execution Idle Time", fontsize=35)
    ax1.set_xlabel("Percentile (%)", fontsize=35)
    ax1.set_ylabel("Time (minutes)", fontsize=35)
    ax1.tick_params(axis="x", labelsize=35)
    ax1.tick_params(axis="y", labelsize=35)
    ax1.legend(fontsize=30, loc="upper left")
    # ax1.set_yscale("log")
    # ax1.yaxis.set_major_locator(mtick.LogLocator(base=10))
    # ax1.yaxis.set_major_formatter(mtick.LogFormatterMathtext(base=10))

    ax2 = axes[1]
    for col in ["Non-GPU","GPU-Enabled"]:
        ax2.plot(post_tbl2["Percentile (%)"], post_tbl2[col], marker="o", markersize=14, linewidth=5, label=col)
    # ax2.set_title(f"{title_prefix} Post-Execution Idle (Percentiles)", fontsize=35)
    ax2.set_title("Post-Execution Idle Time", fontsize=35)
    ax2.set_xlabel("Percentile (%)", fontsize=35)
    ax2.set_ylabel("Time (minutes)", fontsize=35)
    ax2.tick_params(axis="x", labelsize=35)
    ax2.tick_params(axis="y", labelsize=35)
    # ax2.legend(fontsize=30, loc="upper left")
    # ax2.set_yscale("log")
    # ax2.yaxis.set_major_locator(mtick.LogLocator(base=10))
    # ax2.yaxis.set_major_formatter(mtick.LogFormatterMathtext(base=10))

    plt.tight_layout()
    plt.show()

    return pre_tbl2, post_tbl2
# =========================================================
# [E] 고정 3세트(Base + Sensitivity 2개)로 그림 그리기
#   - Base:        (0GPU: CPU10,MEM15) / (GPU: CPU10,MEM20,GPU=0)
#   - Sensitivity: (15,20) / (15,20,0)
#   - Sensitivity: (20,25) / (20,25,0)
# =========================================================

# ✅ 비교할 threshold 세트 정의
fixed_sets = [
    {"name":"BASE",          "CPU0":10, "MEM0":15, "CPUG":10, "MEMG":20, "GPUG":0.0},
    {"name":"SENS1_(10,20)", "CPU0": 5, "MEM0":10, "CPUG":10, "MEMG":15, "GPUG":0.0},
    {"name":"SENS2_(15,20)", "CPU0":15, "MEM0":20, "CPUG":15, "MEMG":20, "GPUG":0.0},
    {"name":"SENS3_(20,25)", "CPU0":20, "MEM0":25, "CPUG":20, "MEMG":25, "GPUG":0.0},
]


# (선택) 각 세트별 요약 테이블을 모아두고 싶으면
all_pre_tbl = {}
all_post_tbl = {}

for i, s in enumerate(fixed_sets):
    CPU0, MEM0, CPUG, MEMG, GPUG = s["CPU0"], s["MEM0"], s["CPUG"], s["MEMG"], s["GPUG"]
    tag = s["name"]

    title = f"[{tag}] 0GPU: CPU<={CPU0}, MEM<={MEM0} | GPU: CPU<={CPUG}, MEM<={MEMG}, GPU<=0"

    pre_df, post_df, _, _ = build_pre_post(normal_data_sorted, CPU0, MEM0, CPUG, MEMG, GPUG)

    if pre_df is None or post_df is None or len(post_df) == 0:
        print(title, "-> No valid post-idle samples (USED->IDLE(end))")
        continue

    print(title, "| post n =", len(post_df))

    pre_tbl2, post_tbl2 = plot_2panel(pre_df, post_df, title_prefix=title)

    display(pre_tbl2)
    display(post_tbl2)
#     all_pre_tbl[tag] = pre_tbl2
#     all_post_tbl[tag] = post_tbl2

# # (선택) 결과 테이블 확인
# display(all_pre_tbl["BASE"])
# display(all_post_tbl["BASE"])
# display(all_pre_tbl["SENS1_(10,20)"])
# display(all_post_tbl["SENS1_(10,20)"])
# display(all_pre_tbl["SENS2_(15,20)"])
# display(all_post_tbl["SENS2_(15,20)"])
# display(all_pre_tbl["SENS3_(20,25)"])
# display(all_post_tbl["SENS3_(20,25)"])


In [ ]:
_mf_capture_cell(40)


In [ ]:
(_MF_OUTPUT/'execution_scope.json').write_text(_mf_json.dumps({'mode':_MF_MODE,'source_notebook':'04_Analysis/section4/Run03_IdleDistribution.ipynb','excluded_original_code_cells':[],'status':'EXECUTED_SCOPE_COMPLETE'},indent=2,ensure_ascii=False))
